# SP-05 Conservative Risk-Aware Policy — `05_conservative_policy.ipynb`

Fifth and last specialised policy of the **Personalized AI Trading Agent**. A separate Meta-RL model will later choose among the five policies; **this model does not select strategies**. It returns one *proposed* action and a confidence for one state.

| Item | Value |
|---|---|
| Model | SP-05 Conservative Risk-Aware Policy (GRU/LSTM encoder + static branch + dense layer + 4-action head) |
| Action space (shared) | `0 = HOLD`, `1 = BUY`, `2 = SELL`, `3 = EXIT` |
| Main artifact | `conservative_policy.pt` (+ `conservative_config.json`, `feature_schema.json`, `preprocessing.pkl`, `evaluation_metrics.json`, `equity_curve.csv`) |
| Execution convention | decision at the **close** of bar *t*, fill at the **open** of bar *t+1* |
| Hard safety | `sp05_risk.RiskController`, a deterministic gate with **no learned parameters** and **final authority** over execution |

> **Research and paper-trading system only.** "Conservative" and "risk-aware" are design intentions, not guarantees. Nothing here claims capital protection, and the stress tests report where protection fails (for example, no controller can prevent an overnight gap loss on exposure that is already open).

### Two separate things, evaluated separately
1. **The learned policy** (neural network): *proposes* HOLD / BUY / SELL / EXIT. It prefers HOLD when evidence is weak, avoids entries in prohibited conditions and proposes exits as risk rises. It is trained by supervised learning on labels from an explicit rule.
2. **The risk gate** (`RiskController`): plain deterministic code. It blocks entries that violate risk limits (volatility, abnormal gaps/moves, market stress, drawdown halt, exposure budget, size, shorts) and **forces EXIT** on hard stop-loss, maximum holding period or portfolio-drawdown flatten. The neural network can never override it, and de-risking proposals are never blocked.

The backtests report *policy + gate*, *policy alone*, *the gate with a naive always-long policy* and a *simple rule-based risk controller*, so you can see what the network adds and what the gate adds.

### Labels are proxy labels, not human demonstrations
By default no genuine trader actions exist, so the labels come from the explicit, decision-time-only **conservative objective** in section 5 (strategy imitation). They are **not** human demonstrations, and they never use hindsight: no future price touches a label. If you have real trader actions, set `CONFIG['trader_labels_csv']`; the reports then label the run as behaviour cloning.

### Notebook map
1 Install · 2 Configuration · 3 Data · 4 Features, adapter and risk module · 5 Conservative objective, labels and dataset · 6 Model · 7 Training · 8 Calibration and classification metrics · 9 Backtest with and without the gate, stress tests and failure analysis · 10 Plots · 11 Export · 12 Inference demo · 13 Assumptions, limitations and Meta-RL integration contract

## 1. Install
Colab already ships PyTorch, pandas, NumPy, scikit-learn and Matplotlib. Only `yfinance` is upgraded. All indicators are implemented manually (no `pandas-ta`, which is fragile on new NumPy/Python versions).

In [ ]:
import subprocess, sys

def _pip_install(*packages):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages])
    except Exception as exc:  # noqa: BLE001 - installation problems must not hide the real error later
        print(f"WARNING: pip install {packages} failed: {exc!r}. Continuing with the installed version, if any.")

_pip_install("yfinance")

## 2. Configuration
All knobs live in `CONFIG`.

* **Splits:** chronological by calendar date; `strict_window_isolation=True` makes every validation/test window start inside its split.
* **Conservative objective** (`CONFIG['objective']`, `CONFIG['soft_limits']`) defines the labels; its trade-off between returns, drawdown and turnover is explained in section 5.
* **Hard limits** (`CONFIG['risk_limits']`) configure the deterministic `RiskController`. They are deliberately *looser* than the soft limits in the labels, so a well-trained policy acts first and the gate is the backstop.
* **Backtest capital rules:** `initial_capital`, `position_fraction`, `max_position_size`, `max_gross_exposure` (physical cap used when no gate is applied), fees, slippage and `confidence_sizing` (smaller entries when confidence is lower).
* **Long/short** is explicit through `allow_short`.

In [ ]:
import os, json, math, time, random, pickle, warnings, platform, importlib, shutil, copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as Fnn
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             precision_recall_fscore_support, confusion_matrix)

warnings.filterwarnings("ignore", category=FutureWarning)

CONFIG = {
    "seed": 42,
    # ---- data ------------------------------------------------------------
    "symbols": ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "JPM", "XOM", "CVX", "JNJ", "PG", "KO", "PEP", "WMT", "V", "HD"],
    "benchmark": "SPY",
    "start_date": "2010-01-01",
    "end_date": "2025-12-31",          # exclusive
    "min_history_rows": 800,           # symbols with fewer clean daily bars are skipped (and reported)
    "local_csv_dir": None,             # optional fallback: <dir>/<SYMBOL>.csv with Date,Open,High,Low,Close,Volume (real data only)
    "trader_labels_csv": None,         # optional genuine labels: columns symbol,date,action,position[,risk_tolerance,patience,aggressiveness]
    # ---- chronological split (calendar dates) ----------------------------
    "train_end": "2020-12-31",
    "val_end": "2022-12-31",
    # ---- sequence / policy -----------------------------------------------
    "seq_len": 30,
    "strict_window_isolation": True,
    "allow_short": True,
    # ---- conservative objective (labels, decision-time only) --------------
    "objective": {"w_return": 1.0,          # weight of the risk-adjusted trend evidence ("edge")
                  "w_drawdown": 1.0,        # weight of the risk score (volatility percentile + drawdown depth)
                  "w_turnover": 0.3,        # cost charged per new position (discourages churn)
                  "entry_threshold": 0.3,   # utility an entry must reach
                  "exit_threshold": -0.25,  # utility below which a held position is closed
                  "short_penalty": 0.2},    # extra utility shorts must clear
    "soft_limits": {"portfolio_drawdown": 0.05, "gross_exposure": 0.50, "recent_loss": 0.04, "vol_exit_percentile": 0.90,
                    "abnormal_ret_z": 3.0, "gap_atr": 2.5, "market_stress_dd": 0.10},
    "hold_sample_max": 60,               # holding duration sampled for position-conditioned training states
    "eval_horizon": 10,                  # bars for the FUTURE-return column used only in evaluation tables
    # ---- backtest --------------------------------------------------------
    "initial_capital": 100000.0,
    "dd_lookback_bars": 120,           # portfolio drawdown = fall below the TRAILING high of this many bars (an all-time peak would lock the account out after a loss)
    "position_fraction": 0.10,         # target notional per new position, fraction of equity
    "max_position_size": 0.20,         # hard cap per position, fraction of equity (applied as min with the target)
    "max_gross_exposure": 1.0,         # sum |notional| <= this * equity (no leverage by default)
    "min_trade_fraction": 0.005,       # smaller entries are skipped
    "fee_bps": 5.0,                    # per side, on traded notional
    "slippage_bps": 5.0,               # per side, adverse price move on every fill
    "short_borrow_bps_annual": 50.0,
    "risk_free_annual": 0.0,
    "confidence_sizing": True,         # scale entry size by calibrated confidence (smaller when less sure)
    # ---- hard limits for the deterministic RiskController (independent of the network) ----
    "risk_limits": {"max_position_fraction": 0.10, "max_gross_exposure": 0.60, "position_stop_loss": 0.05, "max_holding_bars": 120,
                    "halt_drawdown": 0.08, "flatten_drawdown": 0.15, "recent_loss_halt": 0.06, "max_vol_percentile": 0.90,
                    "max_abs_gap_atr": 3.0, "max_abnormal_ret_z": 4.0, "market_stress_dd": 0.12, "min_entry_confidence": 0.50},
    # ---- model / training -----------------------------------------------
    "model": {"encoder": "gru", "hidden_size": 64, "num_layers": 2, "dropout": 0.25,
              "static_hidden": 16, "head_hidden": 64},
    "train": {"epochs": 40, "batch_size": 512, "lr": 1e-3, "weight_decay": 1e-4, "patience": 7,
              "grad_clip": 1.0, "input_noise_std": 0.02, "max_class_weight": 20.0, "scaler_clip": 5.0},
    "artifact_dir": "/content/sp05_conservative_artifacts" if Path("/content").exists() else "./sp05_conservative_artifacts",
}

assert pd.Timestamp(CONFIG["start_date"]) < pd.Timestamp(CONFIG["train_end"]) < pd.Timestamp(CONFIG["val_end"]) < pd.Timestamp(CONFIG["end_date"]), \
    "dates must satisfy start < train_end < val_end < end"

def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ART_DIR = Path(CONFIG["artifact_dir"]); ART_DIR.mkdir(parents=True, exist_ok=True)

print(f"python {platform.python_version()} | torch {torch.__version__} | numpy {np.__version__} | pandas {pd.__version__}")
print(f"device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else " (CPU fallback)"))
print(f"artifact dir: {ART_DIR.resolve()}")

## 3. Data download
Daily OHLCV from Yahoo Finance via `yfinance` with **adjusted** prices (`auto_adjust=True`, so Open/High/Low/Close are all split/dividend adjusted). Failures are retried, then reported explicitly: a failed symbol is **skipped, never fabricated**. If the benchmark cannot be downloaded or fewer than three symbols succeed, the notebook stops with a clear error.

Cleaning rules (all counted in the report below): duplicate dates removed, non-positive or missing OHLC rows dropped, rows with `High < Low` dropped, negative volume set to missing. Missing volume is kept as missing (never invented) and handled later by explicit train-median imputation.

In [ ]:
import yfinance as yf

OHLCV = ["Open", "High", "Low", "Close", "Volume"]

def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df = df.copy(); df.columns = df.columns.get_level_values(0)
    return df

def download_symbol(sym: str, start: str, end: str, retries: int = 3, local_dir=None):
    """Return (raw_df, source). Raises RuntimeError with the reason if nothing real can be obtained."""
    last_err = "unknown error"
    for attempt in range(1, retries + 1):
        try:
            df = yf.download(sym, start=start, end=end, interval="1d", auto_adjust=True, progress=False, threads=False)
            df = _flatten_columns(df) if df is not None else None
            if df is not None and len(df) > 0:
                return df, "yfinance"
            last_err = "empty result"
        except Exception as exc:  # noqa: BLE001
            last_err = repr(exc)
        time.sleep(1.5 * attempt)
    if local_dir:
        path = Path(local_dir) / f"{sym}.csv"
        if path.exists():
            df = pd.read_csv(path, index_col=0, parse_dates=True)
            return df.loc[start:end], f"local_csv:{path}"
    raise RuntimeError(f"{sym}: download failed after {retries} attempts ({last_err})")

def clean_ohlcv(df: pd.DataFrame, sym: str):
    missing = [c for c in OHLCV if c not in df.columns]
    if missing:
        raise ValueError(f"{sym}: missing columns {missing}")
    stats = {"symbol": sym, "raw_rows": int(len(df))}
    df = df[OHLCV].apply(pd.to_numeric, errors="coerce")
    idx = pd.to_datetime(df.index)
    if getattr(idx, "tz", None) is not None:
        idx = idx.tz_localize(None)
    df.index = pd.DatetimeIndex(idx).normalize()
    df.index.name = "Date"
    stats["duplicate_dates_removed"] = int(df.index.duplicated().sum())
    df = df[~df.index.duplicated(keep="last")].sort_index()
    price = ["Open", "High", "Low", "Close"]
    bad = df[price].isna().any(axis=1) | (df[price] <= 0).any(axis=1) | (df["High"] < df["Low"])
    stats["bad_price_rows_dropped"] = int(bad.sum())
    df = df.loc[~bad].copy()
    df.loc[df["Volume"] < 0, "Volume"] = np.nan
    stats["missing_volume_rows"] = int(df["Volume"].isna().sum())
    stats["zero_volume_rows"] = int((df["Volume"] == 0).sum())
    stats["extreme_moves_gt_40pct"] = int((df["Close"].pct_change().abs() > 0.40).sum())
    stats["max_gap_days"] = int(df.index.to_series().diff().dt.days.max()) if len(df) > 1 else 0
    stats["rows"] = int(len(df))
    stats["first_date"] = str(df.index.min().date()) if len(df) else None
    stats["last_date"] = str(df.index.max().date()) if len(df) else None
    return df, stats

download_log, raw_data = [], {}

# benchmark first: it is mandatory
try:
    _b, _src = download_symbol(CONFIG["benchmark"], CONFIG["start_date"], CONFIG["end_date"], local_dir=CONFIG["local_csv_dir"])
    bench_df, _st = clean_ohlcv(_b, CONFIG["benchmark"])
    _st.update(source=_src, status="ok"); download_log.append(_st)
except Exception as exc:  # noqa: BLE001
    raise RuntimeError(f"Benchmark {CONFIG['benchmark']} could not be obtained, cannot continue: {exc}") from exc

for sym in CONFIG["symbols"]:
    try:
        _raw, _src = download_symbol(sym, CONFIG["start_date"], CONFIG["end_date"], local_dir=CONFIG["local_csv_dir"])
        _df, _st = clean_ohlcv(_raw, sym)
        if len(_df) < CONFIG["min_history_rows"]:
            raise ValueError(f"only {len(_df)} clean rows (< min_history_rows={CONFIG['min_history_rows']})")
        _st.update(source=_src, status="ok")
        raw_data[sym] = _df
    except Exception as exc:  # noqa: BLE001
        _st = {"symbol": sym, "status": f"SKIPPED: {exc}"}
        print(f"WARNING: {sym} skipped -> {exc}")
    download_log.append(_st)

if len(raw_data) < 3:
    raise RuntimeError(f"Only {len(raw_data)} symbols downloaded successfully; need at least 3. Check the network / symbols.")

report = pd.DataFrame(download_log).set_index("symbol")
print(f"\nSymbols used ({len(raw_data)}): {list(raw_data)}")
print(f"Symbols skipped: {[s for s in CONFIG['symbols'] if s not in raw_data] or 'none'}")
report

## 4. Features, adapter and the risk module
Two small modules are written to disk by the next cells (both ship with the artifacts, neither touches the network): `sp05_features.py` (features, adapter, action masks) and `sp05_risk.py` (the deterministic gate).

**Shared contract (identical to the other policies)**
* **Sequence, indices 0–29:** `ret_1, logret_1, ret_5, ret_10, ret_20, ret_60` · `rsi_14, macd_pct, macd_signal_pct, macd_hist_pct` · `close_ema20, close_ema50, ema20_ema50, close_sma200, sma50_sma200` · `adx_14, di_diff` · `atr_pct, rv_20, rv_60` · `vol_chg_1, vol_ratio_20, vol_z_60` · `hl_range, gap_open, close_loc` · `rel_strength_20, rel_strength_60, bench_ret_20, bench_rv_20` (returns, realised volatility, ATR, trend strength, momentum, RSI, volume, gap, benchmark-relative).
* **Static, indices 0–7:** position one-hot `[flat, long, short]`, profile `[risk_tolerance, patience, aggressiveness]` (trader risk tolerance), `unrealized_pnl_scaled`, `holding_bars_scaled`.

**SP-05 risk features (13, sequence indices 30–42), produced by `RiskFeatureAdapter` from stock + benchmark OHLCV**

| Feature | Definition (all causal) |
|---|---|
| `atr_pct_pctile_252`, `rv20_pctile_252` | percentile rank of ATR% / 20-bar realised volatility within its trailing 252 values (volatility percentile) |
| `dd_from_high_60`, `dd_from_high_252` | close / trailing max − 1 |
| `maxdd_60` | worst peak-to-trough decline inside the trailing 60 bars (rolling maximum drawdown) |
| `gap_abs_atr` | \|open − previous close\| / ATR at the previous close (gap size) |
| `abnormal_ret_z`, `range_z`, `vol_anomaly_z` | today's return / bar range / volume versus the mean and std of the **previous** 60 bars (abnormal movement, volume anomaly) |
| `bb_width_20` | Bollinger Band width = 4·std20 / mean20 |
| `down_vol_20` | annualised downside deviation over 20 bars |
| `bench_dd_60`, `bench_rv20_pctile_252` | benchmark drawdown and volatility percentile (market stress) |

**SP-05 static extras (indices 8–11):** `portfolio_drawdown_scaled` (portfolio equity below its trailing 120-bar high), `gross_exposure_scaled`, `position_size_scaled` (this position's share of equity, 0 when flat), `recent_loss_scaled` (equity fall over about 20 bars). The caller passes raw fractions; scaling is inside the policy. Every extra defaults to 0.

**Adapter contract:** the Meta-RL environment keeps sending the shared state; for SP-05 it also passes `risk_features` (L × 13) from `RiskFeatureAdapter().adapt_state(state, ohlcv, benchmark)`, or one pre-extended 43-column sequence, plus the four portfolio numbers above.

In [ ]:
%%writefile sp05_features.py
"""SP-05 Conservative Risk-Aware Policy - shared constants and causal feature engineering.

Schema family rule: the first 30 sequence features and the first 8 static features are identical to the other
policies (shared contract). SP-05 adds 13 risk-oriented sequence features (produced by `RiskFeatureAdapter`
from OHLCV + benchmark, passed as `risk_features` or appended to `sequence`) and 4 portfolio-level static
inputs (portfolio drawdown, gross exposure, position size, recent loss; all default to 0).
Every feature at bar t uses data up to and including the close of bar t only.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

# ----------------------------------------------------------------------------
# Common action space (shared by all five specialised policies)
# ----------------------------------------------------------------------------
HOLD, BUY, SELL, EXIT = 0, 1, 2, 3
ACTION_NAMES = ["HOLD", "BUY", "SELL", "EXIT"]
N_ACTIONS = 4

# Position state: +1 long, 0 flat, -1 short. One-hot order below is part of the schema.
POSITION_ORDER = (0, 1, -1)                      # one-hot index -> position value
POSITION_NAMES = ["pos_flat", "pos_long", "pos_short"]

# Trader-profile features, each in [0, 1].
PROFILE_FEATURES = ["risk_tolerance", "patience", "aggressiveness"]
DEFAULT_PROFILE = (0.5, 0.5, 0.5)

STATIC_FEATURE_NAMES = POSITION_NAMES + PROFILE_FEATURES   # shared prefix: 3 + 3 = 6 dims

# Sequential market features: exact column order of the model input (30 dims).
BASE_FEATURE_COLUMNS = [
    # returns
    "ret_1", "logret_1", "ret_5", "ret_10", "ret_20", "ret_60",
    # momentum oscillators
    "rsi_14", "macd_pct", "macd_signal_pct", "macd_hist_pct",
    # moving-average relationships
    "close_ema20", "close_ema50", "ema20_ema50", "close_sma200", "sma50_sma200",
    # trend strength
    "adx_14", "di_diff",
    # volatility
    "atr_pct", "rv_20", "rv_60",
    # volume
    "vol_chg_1", "vol_ratio_20", "vol_z_60",
    # candle shape
    "hl_range", "gap_open", "close_loc",
    # relative strength vs benchmark and market context
    "rel_strength_20", "rel_strength_60", "bench_ret_20", "bench_rv_20",
]
# SP-05 additions (appended after the 30 shared features; see RiskFeatureAdapter)
EXTRA_FEATURE_COLUMNS = [
    "atr_pct_pctile_252", "rv20_pctile_252", "dd_from_high_60", "dd_from_high_252", "maxdd_60",
    "gap_abs_atr", "abnormal_ret_z", "range_z", "vol_anomaly_z", "bb_width_20", "down_vol_20",
    "bench_dd_60", "bench_rv20_pctile_252",
]
FEATURE_COLUMNS = BASE_FEATURE_COLUMNS + EXTRA_FEATURE_COLUMNS
N_BASE_FEATURES = len(BASE_FEATURE_COLUMNS)
N_FEATURES = len(FEATURE_COLUMNS)
N_BASE_STATIC = len(STATIC_FEATURE_NAMES)
SHARED_STATIC_NAMES = list(STATIC_FEATURE_NAMES) + ["unrealized_pnl_scaled", "holding_bars_scaled"]   # identical to SP-02/03/04 (8 dims)
EXTRA_STATIC_NAMES = ["unrealized_pnl_scaled", "holding_bars_scaled",
                      "portfolio_drawdown_scaled", "gross_exposure_scaled", "position_size_scaled", "recent_loss_scaled"]
STATIC_FEATURE_NAMES = STATIC_FEATURE_NAMES + EXTRA_STATIC_NAMES
N_STATIC = len(STATIC_FEATURE_NAMES)
N_SHARED_STATIC = len(SHARED_STATIC_NAMES)
PNL_SCALE, PNL_CLIP = 0.10, 3.0       # unrealized_pnl_scaled = clip(pnl / 0.10, -3, 3)
HOLD_SCALE, HOLD_CAP = 20.0, 60       # holding_bars_scaled   = min(bars, 60) / 20
PDD_SCALE, PDD_CLIP = 0.10, 3.0       # portfolio_drawdown_scaled = clip(drawdown / 0.10, 0, 3)  (drawdown >= 0: below the trailing high)
GROSS_CLIP = 2.0                      # gross_exposure_scaled = clip(gross / equity, 0, 2)
SIZE_SCALE, SIZE_CLIP = 0.20, 3.0     # position_size_scaled = clip(size / 0.20, 0, 3)  (0 when flat)
LOSS_SCALE, LOSS_CLIP = 0.05, 3.0     # recent_loss_scaled = clip(loss / 0.05, 0, 3)  (loss >= 0: equity fall over ~20 bars)
RISK_MIN_HISTORY = 290
WARMUP_BARS = 290          # percentile ranks over 252 bars of a 20-bar volatility need ~272 bars
OHLCV_COLUMNS = ["Open", "High", "Low", "Close", "Volume"]


# ----------------------------------------------------------------------------
# Action validity and position transitions
# ----------------------------------------------------------------------------
def valid_action_mask(position, allow_short: bool = True) -> np.ndarray:
    """Boolean (B, 4) mask of valid actions given the current position.

    flat  -> HOLD, BUY (open long), SELL (open short, only if allow_short)
    long  -> HOLD, EXIT
    short -> HOLD, EXIT
    """
    pos = np.asarray(position).astype(int).reshape(-1)
    mask = np.zeros((pos.size, N_ACTIONS), dtype=bool)
    mask[:, HOLD] = True
    flat = pos == 0
    mask[flat, BUY] = True
    mask[flat, SELL] = bool(allow_short)
    mask[~flat, EXIT] = True
    return mask


def apply_action(position, action) -> np.ndarray:
    """Return the new position after executing (already validated) actions."""
    pos = np.asarray(position).astype(int).reshape(-1)
    act = np.asarray(action).astype(int).reshape(-1)
    new = pos.copy()
    new[(act == BUY) & (pos == 0)] = 1
    new[(act == SELL) & (pos == 0)] = -1
    new[(act == EXIT) & (pos != 0)] = 0
    return new


def profile_array(profile, batch_size: int) -> np.ndarray:
    """Normalise a profile (dict, sequence, (B,3) array or None) to float32 (B, 3)."""
    if profile is None:
        profile = DEFAULT_PROFILE
    if isinstance(profile, dict):
        profile = [profile.get(k, d) for k, d in zip(PROFILE_FEATURES, DEFAULT_PROFILE)]
    arr = np.asarray(profile, dtype=np.float32)
    if arr.ndim == 1:
        arr = np.tile(arr[None, :], (batch_size, 1))
    if arr.shape != (batch_size, len(PROFILE_FEATURES)):
        raise ValueError(f"profile must have shape (3,) or ({batch_size}, 3); got {arr.shape}")
    if np.any(arr < 0) or np.any(arr > 1) or not np.all(np.isfinite(arr)):
        raise ValueError("profile values must be finite and within [0, 1]")
    return arr


def encode_static(position, profile, unrealized_pnl=None, holding_bars=None, portfolio_drawdown=None,
                  gross_exposure=None, position_size=None, recent_loss=None) -> np.ndarray:
    """Static input (B, 12): [flat, long, short] + [risk, patience, aggressiveness] + [pnl, hold] (shared 8 dims)
    + [portfolio_drawdown, gross_exposure, position_size, recent_loss] (SP-05 extras).

    unrealized_pnl     : position-signed fractional P&L since entry (ignored when flat)
    holding_bars       : bars held, entry bar = 1 (ignored when flat)
    portfolio_drawdown : fraction the portfolio equity is below its TRAILING ~120-bar high (0.04 = 4% below); scalar or per state
    gross_exposure     : gross position notional / equity (portfolio level); scalar or per state
    position_size      : this position's notional / equity (ignored when flat)
    recent_loss        : fall of portfolio equity over about the last 20 bars as a fraction (0 if equity rose)
    """
    pos = np.asarray(position).astype(int).reshape(-1)
    if not np.all(np.isin(pos, POSITION_ORDER)):
        raise ValueError("position must be in {-1, 0, +1}")
    n = pos.size
    onehot = np.stack([(pos == p) for p in POSITION_ORDER], axis=1).astype(np.float32)

    def _vec(v, name):
        if v is None:
            return np.zeros(n, dtype=np.float32)
        a = np.asarray(v, dtype=np.float32).reshape(-1)
        if a.size == 1 and n > 1:
            a = np.repeat(a, n)
        if a.size != n or not np.all(np.isfinite(a)):
            raise ValueError(f"{name} must be a finite scalar or have one entry per state")
        return a

    flat = pos == 0
    pnl = np.where(flat, 0.0, np.clip(_vec(unrealized_pnl, "unrealized_pnl") / PNL_SCALE, -PNL_CLIP, PNL_CLIP))
    hold = np.where(flat, 0.0, np.minimum(np.maximum(_vec(holding_bars, "holding_bars"), 0.0), HOLD_CAP) / HOLD_SCALE)
    pdd = np.clip(_vec(portfolio_drawdown, "portfolio_drawdown") / PDD_SCALE, 0.0, PDD_CLIP)
    gross = np.clip(_vec(gross_exposure, "gross_exposure"), 0.0, GROSS_CLIP)
    size = np.where(flat, 0.0, np.clip(_vec(position_size, "position_size") / SIZE_SCALE, 0.0, SIZE_CLIP))
    loss = np.clip(_vec(recent_loss, "recent_loss") / LOSS_SCALE, 0.0, LOSS_CLIP)
    cols = [pnl, hold, pdd, gross, size, loss]
    return np.concatenate([onehot, profile_array(profile, n)] + [c[:, None] for c in cols], axis=1).astype(np.float32)


# ----------------------------------------------------------------------------
# Indicators (manual implementations, causal)
# ----------------------------------------------------------------------------
def _pct(s: pd.Series, n: int) -> pd.Series:
    return s / s.shift(n) - 1.0


def _wilder(s: pd.Series, n: int) -> pd.Series:
    return s.ewm(alpha=1.0 / n, adjust=False, min_periods=n).mean()


def rsi(close: pd.Series, n: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = (-delta).clip(lower=0.0)
    rs = _wilder(gain, n) / _wilder(loss, n)
    return 100.0 - 100.0 / (1.0 + rs)


def true_range(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    prev = close.shift(1)
    return pd.concat([high - low, (high - prev).abs(), (low - prev).abs()], axis=1).max(axis=1)


def adx_di(high: pd.Series, low: pd.Series, close: pd.Series, n: int = 14):
    """Wilder ADX with +DI / -DI."""
    up = high.diff()
    down = -low.diff()
    plus_dm = pd.Series(np.where((up > down) & (up > 0), up, 0.0), index=high.index)
    minus_dm = pd.Series(np.where((down > up) & (down > 0), down, 0.0), index=high.index)
    atr = _wilder(true_range(high, low, close), n)
    plus_di = 100.0 * _wilder(plus_dm, n) / atr
    minus_di = 100.0 * _wilder(minus_dm, n) / atr
    dx = 100.0 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0.0, np.nan)
    return _wilder(dx, n), plus_di, minus_di


# ----------------------------------------------------------------------------
# Risk features (OHLCV + benchmark) and the documented adapter
# ----------------------------------------------------------------------------
def _pctile_rank(s: pd.Series, n: int) -> pd.Series:
    """Percentile rank (0..1) of the latest value within its trailing n values (including itself). Causal."""
    return s.rolling(n, min_periods=n).apply(lambda x: float((x <= x[-1]).mean()), raw=True)


def _rolling_maxdd(c: pd.Series, n: int) -> pd.Series:
    """Worst peak-to-trough decline (<= 0) inside the trailing n bars. Causal."""
    def f(x):
        return float((x / np.maximum.accumulate(x) - 1.0).min())
    return c.rolling(n, min_periods=n).apply(f, raw=True)


def risk_features(ohlcv: pd.DataFrame, benchmark: pd.DataFrame) -> pd.DataFrame:
    """The 13 SP-05 risk features (EXTRA_FEATURE_COLUMNS). Causal; z-scores compare bar t with the PREVIOUS 60 bars."""
    df = ohlcv[OHLCV_COLUMNS].astype(float)
    o, h, l, c, v = df["Open"], df["High"], df["Low"], df["Close"], df["Volume"]
    vz = v.where(v > 0)
    ret = c / c.shift(1) - 1.0
    tr = true_range(h, l, c)
    atr14 = _wilder(tr, 14)
    atr_pct = atr14 / c
    lr = np.log(c).diff()
    rv20 = lr.rolling(20).std() * np.sqrt(252.0)
    out = pd.DataFrame(index=df.index)
    out["atr_pct_pctile_252"] = _pctile_rank(atr_pct, 252)            # volatility percentile (ATR based)
    out["rv20_pctile_252"] = _pctile_rank(rv20, 252)                  # volatility percentile (realised)
    out["dd_from_high_60"] = c / c.rolling(60).max() - 1.0
    out["dd_from_high_252"] = c / c.rolling(252).max() - 1.0
    out["maxdd_60"] = _rolling_maxdd(c, 60)                           # rolling maximum drawdown
    out["gap_abs_atr"] = (o - c.shift(1)).abs() / atr14.shift(1)      # gap size in ATRs (abnormal gap indicator)
    sd60 = ret.shift(1).rolling(60).std().replace(0.0, np.nan)
    out["abnormal_ret_z"] = ret / sd60                                # abnormal movement vs the previous 60 bars
    hl = (h - l) / c
    out["range_z"] = (hl - hl.shift(1).rolling(60).mean()) / hl.shift(1).rolling(60).std().replace(0.0, np.nan)
    out["vol_anomaly_z"] = (vz - vz.shift(1).rolling(60).mean()) / vz.shift(1).rolling(60).std().replace(0.0, np.nan)
    m20, s20 = c.rolling(20).mean(), c.rolling(20).std()
    out["bb_width_20"] = (4.0 * s20) / m20                            # Bollinger Band width
    out["down_vol_20"] = np.sqrt((np.minimum(ret, 0.0) ** 2).rolling(20).mean()) * np.sqrt(252.0)   # downside deviation
    bc = benchmark["Close"].astype(float).reindex(df.index).ffill(limit=3)                 # past-only fill
    out["bench_dd_60"] = bc / bc.rolling(60).max() - 1.0              # market stress: benchmark drawdown
    brv20 = np.log(bc).diff().rolling(20).std() * np.sqrt(252.0)
    out["bench_rv20_pctile_252"] = _pctile_rank(brv20, 252)           # market stress: benchmark volatility percentile
    return out[EXTRA_FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)


class RiskFeatureAdapter:
    """Documented adapter: OHLCV (+ benchmark) history -> the 13 SP-05 risk features, added to a shared state.

    The shared state (sequence (L, 30), position, profile, unrealized_pnl, holding_bars) is unchanged; the adapter
    only adds ``risk_features`` (L, 13). It never downloads data: the caller supplies OHLCV history of at least
    ``min_history`` + seq_len bars ending at the decision bar (for stock and benchmark, same dates).
    """
    min_history = RISK_MIN_HISTORY
    columns = list(EXTRA_FEATURE_COLUMNS)

    def transform(self, ohlcv: pd.DataFrame, benchmark: pd.DataFrame) -> pd.DataFrame:
        return risk_features(ohlcv, benchmark)

    def window(self, ohlcv: pd.DataFrame, benchmark: pd.DataFrame, seq_len: int) -> np.ndarray:
        if len(ohlcv) < self.min_history + seq_len:
            raise ValueError(f"need at least {self.min_history + seq_len} bars of OHLCV history; got {len(ohlcv)}")
        return self.transform(ohlcv, benchmark).iloc[-seq_len:].to_numpy(dtype=np.float32)

    def adapt_state(self, state: dict, ohlcv: pd.DataFrame, benchmark: pd.DataFrame) -> dict:
        seq_len = np.asarray(state["sequence"]).shape[-2]
        return {**state, "risk_features": self.window(ohlcv, benchmark, seq_len)}


# ----------------------------------------------------------------------------
# Feature builder
# ----------------------------------------------------------------------------
def build_features(ohlcv: pd.DataFrame, benchmark: pd.DataFrame) -> pd.DataFrame:
    """Compute causal features for one symbol.

    Parameters
    ----------
    ohlcv : DataFrame indexed by date, columns Open/High/Low/Close/Volume (adjusted prices).
    benchmark : DataFrame indexed by date with at least a Close column.

    Returns a DataFrame with the same index: the OHLCV columns followed by FEATURE_COLUMNS.
    Row t only depends on rows <= t (verified by a perturbation test in the notebook).
    """
    if benchmark is None or "Close" not in benchmark:
        raise ValueError("a benchmark DataFrame with a Close column is required")
    missing = [c for c in OHLCV_COLUMNS if c not in ohlcv.columns]
    if missing:
        raise ValueError(f"ohlcv is missing columns: {missing}")

    df = ohlcv[OHLCV_COLUMNS].astype(float).copy()
    o, h, l, c, v = df["Open"], df["High"], df["Low"], df["Close"], df["Volume"]
    f = pd.DataFrame(index=df.index)

    # returns
    f["ret_1"] = _pct(c, 1)
    f["logret_1"] = np.log(c).diff()
    for n in (5, 10, 20, 60):
        f[f"ret_{n}"] = _pct(c, n)

    # momentum oscillators
    f["rsi_14"] = rsi(c, 14)
    ema12 = c.ewm(span=12, adjust=False, min_periods=12).mean()
    ema26 = c.ewm(span=26, adjust=False, min_periods=26).mean()
    macd = ema12 - ema26
    macd_signal = macd.ewm(span=9, adjust=False, min_periods=9).mean()
    f["macd_pct"] = macd / c
    f["macd_signal_pct"] = macd_signal / c
    f["macd_hist_pct"] = (macd - macd_signal) / c

    # moving-average relationships
    ema20 = c.ewm(span=20, adjust=False, min_periods=20).mean()
    ema50 = c.ewm(span=50, adjust=False, min_periods=50).mean()
    sma50 = c.rolling(50).mean()
    sma200 = c.rolling(200).mean()
    f["close_ema20"] = c / ema20 - 1.0
    f["close_ema50"] = c / ema50 - 1.0
    f["ema20_ema50"] = ema20 / ema50 - 1.0
    f["close_sma200"] = c / sma200 - 1.0
    f["sma50_sma200"] = sma50 / sma200 - 1.0

    # trend strength
    adx, plus_di, minus_di = adx_di(h, l, c, 14)
    f["adx_14"] = adx
    f["di_diff"] = plus_di - minus_di

    # volatility
    f["atr_pct"] = _wilder(true_range(h, l, c), 14) / c
    f["rv_20"] = f["logret_1"].rolling(20).std() * np.sqrt(252.0)
    f["rv_60"] = f["logret_1"].rolling(60).std() * np.sqrt(252.0)

    # volume (zero volume treated as missing for ratios; never fabricated)
    vz = v.where(v > 0)
    f["vol_chg_1"] = np.log(vz / vz.shift(1)).clip(-3.0, 3.0)
    f["vol_ratio_20"] = vz / vz.rolling(20).mean()
    f["vol_z_60"] = (vz - vz.rolling(60).mean()) / vz.rolling(60).std().replace(0.0, np.nan)

    # candle shape
    f["hl_range"] = (h - l) / c
    f["gap_open"] = o / c.shift(1) - 1.0
    f["close_loc"] = (c - l) / (h - l).replace(0.0, np.nan)

    # relative strength vs benchmark (past-only forward fill, at most 3 bars)
    bc = benchmark["Close"].astype(float).reindex(df.index).ffill(limit=3)
    b_ret20 = _pct(bc, 20)
    b_ret60 = _pct(bc, 60)
    f["rel_strength_20"] = f["ret_20"] - b_ret20
    f["rel_strength_60"] = f["ret_60"] - b_ret60
    f["bench_ret_20"] = b_ret20
    f["bench_rv_20"] = np.log(bc).diff().rolling(20).std() * np.sqrt(252.0)

    # ---- SP-05 risk extras (same function the adapter exposes) -------------------------------
    ex = risk_features(df, benchmark)
    for col in EXTRA_FEATURE_COLUMNS:
        f[col] = ex[col]

    f = f[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    return pd.concat([df, f], axis=1)


def scale_features(x, median, mean, std, clip) -> np.ndarray:
    """Single implementation of the preprocessing used in training AND inference.

    Non-finite values -> training-set median, then z-score, then clip to +/- clip.
    """
    x = np.asarray(x, dtype=np.float32)
    x = np.where(np.isfinite(x), x, np.asarray(median, dtype=np.float32))
    x = (x - np.asarray(mean, dtype=np.float32)) / np.asarray(std, dtype=np.float32)
    return np.clip(x, -float(clip), float(clip)).astype(np.float32)


def build_feature_schema(seq_len: int) -> dict:
    """Machine-readable description of the exact model input layout."""
    return {
        "schema_version": "1.0.0",
        "policy_id": "SP-05",
        "sequence_input": {
            "name": "sequence",
            "shape": ["batch", seq_len, N_FEATURES],
            "dtype": "float32",
            "order": "oldest bar first, newest bar last (index -1 is the decision bar)",
            "features": [{"index": i, "name": n} for i, n in enumerate(FEATURE_COLUMNS)],
            "scaling": "NaN -> train median, then (x - mean) / std, then clip to +/- clip (see preprocessing.pkl)",
            "note": "Callers pass RAW (unscaled) features; the policy scales internally.",
        },
        "static_input": {
            "name": "static",
            "shape": ["batch", N_STATIC],
            "extras_note": "indices 0-7 identical to SP-02/03/04; indices 8-11 (portfolio drawdown, gross exposure, position size, recent loss) are SP-05 additions; every extra defaults to 0 when absent",
            "dtype": "float32",
            "features": [{"index": i, "name": n} for i, n in enumerate(STATIC_FEATURE_NAMES)],
            "position_encoding": {"one_hot_order": list(POSITION_NAMES), "position_values": list(POSITION_ORDER)},
            "profile_ranges": {k: [0.0, 1.0] for k in PROFILE_FEATURES},
            "default_profile": dict(zip(PROFILE_FEATURES, DEFAULT_PROFILE)),
        },
        "output": {
            "action_probs": ["batch", N_ACTIONS],
            "action_space": {str(i): n for i, n in enumerate(ACTION_NAMES)},
            "confidence": "temperature-calibrated probability of the chosen action after masking invalid actions",
        },
        "valid_actions": {
            "flat": ["HOLD", "BUY", "SELL (only if allow_short)"],
            "long": ["HOLD", "EXIT"],
            "short": ["HOLD", "EXIT"],
        },
        "execution_convention": "decision at close of bar t; fill at open of bar t+1",
        "compatibility": {
            "family_rule": "every SP-0x policy consumes the shared base features first, then its own extras; extras DIFFER per policy, so select columns by name",
            "shared_sequence_features": list(BASE_FEATURE_COLUMNS),
            "sp05_extra_sequence_features": list(EXTRA_FEATURE_COLUMNS),
            "shared_static_features": list(SHARED_STATIC_NAMES),
            "sp05_extra_static_features": list(STATIC_FEATURE_NAMES[N_SHARED_STATIC:]),
            "state_dict_keys": ["sequence", "position", "profile", "unrealized_pnl (optional)", "holding_bars (optional)", "risk_features (SP-05; from RiskFeatureAdapter)", "portfolio_drawdown (optional)", "gross_exposure (optional)", "position_size (optional)", "recent_loss (optional)"],
            "adapter": {"class": "RiskFeatureAdapter", "input": "stock and benchmark OHLCV history ending at the decision bar (>= seq_len + 290 bars)", "output": "risk_features (L, 13)", "alternative": "pass sequence already extended to 43 columns"},
            "authority": "SP-05 only PROPOSES actions; sp05_risk.RiskController has final authority over execution",
        },
    }


In [ ]:
%%writefile sp05_risk.py
"""SP-05 deterministic risk controller (the hard gate).

This module contains NO neural network and NO learned parameters. The policy (sp05_model.ConservativePolicy) only
PROPOSES an action; ``RiskController`` has final authority over what may execute. Rules use RAW (unscaled) feature
values and portfolio state, are pure functions, and are identical in backtests and in live/paper use.

Rules
-----
Entry blocks (a proposed BUY/SELL becomes HOLD):
  short_not_allowed, drawdown_halt, recent_loss_halt, volatility_prohibited, abnormal_gap_or_move, market_stress,
  gross_exposure_limit, low_confidence_entry, risk_data_missing (fail-safe: unknown risk -> no new entries)
Forced exits (any position becomes EXIT, whatever was proposed):
  position_stop_loss, max_holding_period, drawdown_flatten
De-risking proposals (EXIT) are never blocked. Invalid proposals for the current position become HOLD (invalid_action).
"""
from __future__ import annotations

from dataclasses import asdict, dataclass

import numpy as np

from sp05_features import BUY, EXIT, HOLD, SELL, valid_action_mask

REASON_NAMES = [
    "invalid_action", "short_not_allowed", "position_stop_loss", "max_holding_period", "drawdown_flatten",
    "drawdown_halt", "recent_loss_halt", "volatility_prohibited", "abnormal_gap_or_move", "market_stress",
    "gross_exposure_limit", "low_confidence_entry", "risk_data_missing",
]
FORCED_EXIT_REASONS = ["position_stop_loss", "max_holding_period", "drawdown_flatten"]
REQUIRED_FEATURES = ["atr_pct_pctile_252", "rv20_pctile_252", "gap_abs_atr", "abnormal_ret_z", "bench_dd_60"]


@dataclass
class RiskLimits:
    allow_short: bool = True
    max_position_fraction: float = 0.10     # entry size cap, fraction of equity
    max_gross_exposure: float = 0.60        # sum |notional| / equity after the entry
    position_stop_loss: float = 0.05        # hard per-position loss limit -> forced EXIT
    max_holding_bars: int = 120             # hard time limit -> forced EXIT (0 disables)
    halt_drawdown: float = 0.08             # portfolio drawdown (below the trailing ~120-bar high): no new entries
    flatten_drawdown: float = 0.15          # portfolio drawdown: force EXIT everywhere
    recent_loss_halt: float = 0.06          # ~20-bar equity loss: no new entries
    max_vol_percentile: float = 0.90        # no new entries when ATR or realised-vol percentile is above this
    max_abs_gap_atr: float = 3.0            # no new entries after a gap larger than this many ATRs
    max_abnormal_ret_z: float = 4.0         # no new entries after a move this many sigmas abnormal
    market_stress_dd: float = 0.12          # no new entries when the benchmark is this far below its 60-bar high
    min_entry_confidence: float = 0.50      # no new entries when the proposing policy's confidence is below this

    def to_dict(self) -> dict:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict) -> "RiskLimits":
        return cls(**{k: v for k, v in d.items() if k in cls.__dataclass_fields__})


class RiskController:
    """Deterministic gate: final authority over execution."""

    def __init__(self, limits: RiskLimits | None = None):
        self.limits = limits or RiskLimits()

    @classmethod
    def from_policy_config(cls, config: dict) -> "RiskController":
        return cls(RiskLimits.from_dict(config.get("risk_limits", {})))

    # ------------------------------------------------------------------ batch implementation
    def review_batch(self, proposed, confidence, position, unrealized_pnl, holding_bars, features: dict, portfolio: dict):
        """Review one proposal per row.

        proposed, confidence, position, unrealized_pnl, holding_bars : arrays (B,)
        features  : dict of RAW latest-bar feature arrays (B,) containing REQUIRED_FEATURES
        portfolio : dict with scalars  portfolio_drawdown (>=0), gross_exposure, entry_fraction (planned entry size)

        Returns (final_actions (B,), reasons (B, len(REASON_NAMES)) bool).
        """
        lim = self.limits
        proposed = np.asarray(proposed).astype(int).reshape(-1)
        n = proposed.size
        conf = np.asarray(confidence, dtype=float).reshape(-1)
        pos = np.asarray(position).astype(int).reshape(-1)
        pnl = np.asarray(unrealized_pnl, dtype=float).reshape(-1)
        hold = np.asarray(holding_bars, dtype=float).reshape(-1)
        R = {name: np.zeros(n, dtype=bool) for name in REASON_NAMES}
        final = proposed.copy()

        # 1) proposals that are impossible for the current position never execute
        valid = valid_action_mask(pos, lim.allow_short)
        bad = ~valid[np.arange(n), final]
        R["short_not_allowed"] = bad & (final == SELL) & (pos == 0) & (not lim.allow_short)
        R["invalid_action"] = bad & ~R["short_not_allowed"]
        final[bad] = HOLD

        # 2) entry blocks
        pdd = float(portfolio.get("portfolio_drawdown", 0.0)); gross = float(portfolio.get("gross_exposure", 0.0))
        rloss = float(portfolio.get("recent_loss", 0.0))
        entry = (final == BUY) | (final == SELL)
        with np.errstate(invalid="ignore"):
            vol_pct = np.maximum(np.asarray(features["atr_pct_pctile_252"], float), np.asarray(features["rv20_pctile_252"], float))
            known = np.all([np.isfinite(np.asarray(features[k], float)) for k in REQUIRED_FEATURES], axis=0)
            R["risk_data_missing"] = entry & ~known
            R["drawdown_halt"] = entry & (pdd >= lim.halt_drawdown)
            R["recent_loss_halt"] = entry & (rloss >= lim.recent_loss_halt)
            R["volatility_prohibited"] = entry & (vol_pct > lim.max_vol_percentile)
            R["abnormal_gap_or_move"] = entry & ((np.asarray(features["gap_abs_atr"], float) > lim.max_abs_gap_atr)
                                                 | (np.abs(np.asarray(features["abnormal_ret_z"], float)) > lim.max_abnormal_ret_z))
            R["market_stress"] = entry & (np.asarray(features["bench_dd_60"], float) <= -lim.market_stress_dd)
            R["low_confidence_entry"] = entry & (conf < lim.min_entry_confidence)
        blocked = np.zeros(n, dtype=bool)
        for k in ("risk_data_missing", "drawdown_halt", "recent_loss_halt", "volatility_prohibited", "abnormal_gap_or_move",
                  "market_stress", "low_confidence_entry"):
            blocked |= R[k]
        # gross-exposure budget, most confident entries first
        running = gross; frac = float(portfolio.get("entry_fraction", lim.max_position_fraction))
        frac = min(frac, lim.max_position_fraction)
        for i in np.argsort(-conf):
            if entry[i] and not blocked[i]:
                if running + frac <= lim.max_gross_exposure + 1e-12:
                    running += frac
                else:
                    R["gross_exposure_limit"][i] = True; blocked[i] = True
        final[blocked] = HOLD

        # 3) forced exits for held positions (overrule HOLD and any proposal other than EXIT)
        held = pos != 0
        with np.errstate(invalid="ignore"):
            R["position_stop_loss"] = held & (pnl <= -lim.position_stop_loss)
            R["max_holding_period"] = held & (lim.max_holding_bars > 0) & (hold >= lim.max_holding_bars)
        R["drawdown_flatten"] = held & (pdd >= lim.flatten_drawdown)
        forced = R["position_stop_loss"] | R["max_holding_period"] | R["drawdown_flatten"]
        final[forced] = EXIT
        reasons = np.stack([R[k] for k in REASON_NAMES], axis=1)
        return final, reasons

    # ------------------------------------------------------------------ single-state convenience
    def review(self, proposal: dict, state: dict, feature_names: list) -> dict:
        """Review one policy output. ``state['sequence']`` must contain the RAW features (all 43 columns, or the 30 shared +
        'risk_features'); the latest bar supplies the risk features. Returns a dict with the final action and reasons."""
        seq = np.asarray(state["sequence"], dtype=float)
        extra = state.get("risk_features")
        last = seq[-1] if seq.ndim == 2 else seq[0, -1]
        if extra is not None:
            ex = np.asarray(extra, dtype=float); last = np.concatenate([last, ex[-1] if ex.ndim == 2 else ex[0, -1]])
        feats = {name: np.array([last[i]]) for i, name in enumerate(feature_names) if i < last.size}
        port = {"portfolio_drawdown": state.get("portfolio_drawdown", 0.0) or 0.0, "gross_exposure": state.get("gross_exposure", 0.0) or 0.0,
                "recent_loss": state.get("recent_loss", 0.0) or 0.0}
        final, reasons = self.review_batch([proposal["action"]], [proposal.get("confidence", 1.0)], [state.get("position", 0)],
                                           [state.get("unrealized_pnl", 0.0) or 0.0], [state.get("holding_bars", 0) or 0], feats, port)
        names = [r for r, on in zip(REASON_NAMES, reasons[0]) if on]
        return {"action": int(final[0]), "proposed_action": int(proposal["action"]), "overridden": bool(final[0] != proposal["action"]), "reasons": names}


In [ ]:
import sp05_features, sp05_risk
importlib.reload(sp05_features); importlib.reload(sp05_risk)
from sp05_features import (ACTION_NAMES, HOLD, BUY, SELL, EXIT, N_ACTIONS, FEATURE_COLUMNS, BASE_FEATURE_COLUMNS,
                           EXTRA_FEATURE_COLUMNS, N_FEATURES, N_BASE_FEATURES, N_STATIC, N_SHARED_STATIC, STATIC_FEATURE_NAMES,
                           SHARED_STATIC_NAMES, PROFILE_FEATURES, DEFAULT_PROFILE, RiskFeatureAdapter, risk_features,
                           build_features, build_feature_schema, encode_static, valid_action_mask, apply_action, scale_features)
from sp05_risk import RiskController, RiskLimits, REASON_NAMES, FORCED_EXIT_REASONS, REQUIRED_FEATURES

print(f"{N_FEATURES} model sequence features = {N_BASE_FEATURES} shared + {len(EXTRA_FEATURE_COLUMNS)} risk (via adapter); {N_STATIC} static features = {N_SHARED_STATIC} shared + {N_STATIC - N_SHARED_STATIC} portfolio extras")
for i, name in enumerate(FEATURE_COLUMNS):
    print(f"  seq[{i:2d}] {name}{'   <- risk adapter' if i >= N_BASE_FEATURES else ''}")
for i, name in enumerate(STATIC_FEATURE_NAMES):
    print(f"  static[{i:2d}] {name}{'   <- SP-05 portfolio extra' if i >= N_SHARED_STATIC else ''}")
RISK_LIMITS = RiskLimits(allow_short=CONFIG["allow_short"], **CONFIG["risk_limits"])
print("\nhard risk limits (deterministic gate):", RISK_LIMITS.to_dict())

In [ ]:
H = CONFIG["eval_horizon"]
EVAL_ONLY_COLUMNS = ["fwd_ret_h"]   # FUTURE prices: evaluation tables only, never a model input and never used for labels
assert set(EVAL_ONLY_COLUMNS).isdisjoint(FEATURE_COLUMNS)

def causality_test(sym: str, frac: float = 0.7) -> None:
    ohlcv = raw_data[sym]; n = int(len(ohlcv) * frac)
    base = build_features(ohlcv, bench_df)[FEATURE_COLUMNS].iloc[:n].to_numpy()
    rng = np.random.default_rng(0)
    pert = ohlcv.copy()
    pert.iloc[n:, :] = pert.iloc[n:, :].to_numpy() * rng.uniform(0.3, 3.0, size=pert.iloc[n:].shape)   # scramble the FUTURE
    bench_p = bench_df.copy()
    future = bench_p.index > ohlcv.index[n - 1]
    bench_p.loc[future, :] = bench_p.loc[future].to_numpy() * 2.0
    alt = build_features(pert, bench_p)[FEATURE_COLUMNS].iloc[:n].to_numpy()
    assert np.allclose(base, alt, equal_nan=True, rtol=1e-9, atol=1e-12), f"{sym}: features depend on future data!"

for _s in list(raw_data)[:2]:
    causality_test(_s)
print("Causality test passed: features at row t are unchanged when all data after t is scrambled.")

_sym = list(raw_data)[0]
_full = build_features(raw_data[_sym], bench_df)[EXTRA_FEATURE_COLUMNS].iloc[-30:].to_numpy(dtype=np.float32)
assert np.allclose(RiskFeatureAdapter().window(raw_data[_sym], bench_df, 30), _full, equal_nan=True), "adapter output differs from training features"
print("Adapter parity check passed: RiskFeatureAdapter reproduces the training features exactly.")

frames, feature_report = [], []
for sym, ohlcv in raw_data.items():
    f = build_features(ohlcv, bench_df)
    firsts = [f[c].first_valid_index() for c in FEATURE_COLUMNS]
    if any(x is None for x in firsts):
        print(f"WARNING: {sym} skipped: a feature is entirely missing (history shorter than the 252-bar percentile look-back)")
        continue
    f = f.loc[max(firsts):].copy()                                  # drop warm-up rows only
    nan_before = int(f[FEATURE_COLUMNS].isna().sum().sum())
    f[FEATURE_COLUMNS] = f[FEATURE_COLUMNS].ffill(limit=3)          # past-only fill, at most 3 bars
    nan_after = int(f[FEATURE_COLUMNS].isna().sum().sum())
    f["symbol"] = sym
    f["fwd_ret_h"] = f["Open"].shift(-(H + 1)) / f["Open"].shift(-1) - 1.0      # EVAL ONLY (future)
    f = f.reset_index().rename(columns={"Date": "date", "index": "date"})
    feature_report.append({"symbol": sym, "rows_after_warmup": len(f), "first_date": f["date"].min().date(),
                           "nan_cells_before_ffill": nan_before, "nan_cells_after_ffill": nan_after})
    frames.append(f)
if not frames:
    raise RuntimeError("no symbol has enough history after the warm-up; widen start_date")

panel = pd.concat(frames, ignore_index=True).sort_values(["symbol", "date"]).reset_index(drop=True)
panel["row_in_sym"] = panel.groupby("symbol").cumcount()
print(f"\npanel: {panel.shape[0]:,} rows x {panel.shape[1]} columns, {panel['symbol'].nunique()} symbols, "
      f"{panel['date'].min().date()} -> {panel['date'].max().date()}")
print("Residual NaNs in features are imputed with the TRAIN median at scaling time (the hard gate instead treats missing risk data as 'no new entries').")
pd.DataFrame(feature_report).set_index("symbol")

## 5. The conservative objective, labels and dataset

### 5.1 Decision-time information only (no hindsight)
Every ingredient of a label is known at the close of bar *t*: features, position, unrealized P&L, holding bars, trader profile and the portfolio state (drawdown, gross exposure, recent loss). **No future price is used.** The label rule never "knows" where the market turns, so it cannot create hindsight-perfect exits; it exits late, on average, exactly like a real rule-based risk manager. (`fwd_ret_h` exists only for evaluation tables.)

### 5.2 How returns, drawdown and turnover are traded off
Two decision-time scores summarise a bar:

* **edge** (return evidence) `= 0.6·clip(ret_60 / (rv_60·√(60/252)), ±3) + 0.6·struct`, where `struct = ½·sign(SMA50−SMA200) + ½·sign(close−EMA50)` ∈ [−1, 1]. It is a risk-adjusted trend measure, not a forecast.
* **risk** `= vol_pct + ½·clip(−dd_from_high_60 / 10%, 0, 2)`, where `vol_pct` is the mean of the ATR and realised-volatility percentiles (0…1). It rises with volatility and with depth of the current drawdown.

With the profile-adjusted weights `w_ret = w_return`, `w_dd = w_drawdown·(1.5 − risk_tolerance)`, `w_turn = w_turnover·(0.5 + patience)` and `thr = entry_threshold + 0.5·(1 − aggressiveness)`:

* **Long entry utility** `U_long = w_ret·edge − w_dd·risk − w_turn`; **short entry utility** `U_short = −w_ret·edge − w_dd·risk − w_turn − short_penalty`.
* BUY / SELL (flat state) only if `U ≥ thr` **and** no prohibition applies. Otherwise **HOLD**: weak evidence never forces a trade.
* **Held-position utility** `U_hold = ±w_ret·edge − w_dd·risk`; EXIT when `U_hold ≤ exit_threshold`.

The three terms map to the three goals: `w_return` rewards trend evidence (returns), `w_drawdown` penalises volatile and drawdown-heavy conditions (drawdown awareness) and `w_turnover` plus the entry threshold charge every new position (turnover). Raising `w_drawdown` or `entry_threshold`, or lowering `w_return`, makes the policy trade less and exit sooner. Nothing here proves the weights are good; they are explicit, configurable assumptions.

### 5.3 Soft risk behaviour built into the labels (inside the hard limits)
* **Prohibited entries** → HOLD: volatility percentile `≥ 0.6 + 0.2·risk_tolerance`, abnormal move (`|abnormal_ret_z| ≥ 3`) or gap (`≥ 2.5 ATR`), benchmark 60-bar drawdown deeper than 10%, portfolio drawdown ≥ 5%, gross exposure ≥ 50% or recent loss ≥ 4% (prefer lower exposure).
* **Reduce / close exposure** → EXIT: unrealized loss beyond the profile's soft stop (`1.5% + 3%·risk_tolerance`), low held-position utility, a volatility spike (`vol_pct ≥ 0.9`), an adverse abnormal move against the position (`≥ 3σ`), portfolio drawdown ≥ 5% or a recent loss ≥ 4% while the position is under water.
* **Position rules:** flat → HOLD/BUY/SELL (SELL only if `allow_short`); long or short → HOLD/EXIT. SELL never means "exit", and EXIT while flat is impossible.

### 5.4 Dataset construction
Each bar yields three samples (flat / long / short) with a random profile. For held states the holding duration is sampled and unrealized P&L is computed from real prices as if the position had been filled at the open that many bars ago. The portfolio inputs (drawdown, gross exposure, position size, recent loss) are sampled from broad ranges because a single-symbol price history cannot define them; the backtest later feeds the real portfolio values. Splits are chronological; validation/test windows start inside their split; shuffling only among training samples; the scaler is fit on training-period rows only; class imbalance uses capped inverse-frequency weights.

In [ ]:
D_TRAIN_END = pd.Timestamp(CONFIG["train_end"]); D_VAL_END = pd.Timestamp(CONFIG["val_end"])
SPLIT_BOUNDS = {"train": (None, D_TRAIN_END),
                "val": (D_TRAIN_END + pd.Timedelta(days=1), D_VAL_END),
                "test": (D_VAL_END + pd.Timedelta(days=1), None)}
L = CONFIG["seq_len"]
RAW = {c: panel[c].to_numpy(dtype=float) for c in FEATURE_COLUMNS}               # unscaled, with NaNs
RAW_MAT = panel[FEATURE_COLUMNS].to_numpy(dtype=np.float32)                      # (N, F)
OPEN_ARR = panel["Open"].to_numpy(dtype=float); CLOSE_ARR = panel["Close"].to_numpy(dtype=float)
FWD_RET = panel["fwd_ret_h"].to_numpy(dtype=float)
ROW_IN_SYM = panel["row_in_sym"].to_numpy()
_WIN_START = panel.groupby("symbol")["date"].shift(L - 1)
OBJ, SOFT = CONFIG["objective"], CONFIG["soft_limits"]

def eligible_rows(split: str) -> np.ndarray:
    """Rows whose decision bar is in the split and whose L-bar window is complete (and starts inside the split when
    strict_window_isolation)."""
    start, end = SPLIT_BOUNDS[split]
    ok = _WIN_START.notna().to_numpy().copy()
    if start is not None:
        ok &= (panel["date"] >= start).to_numpy()
        if CONFIG["strict_window_isolation"]:
            ok &= (_WIN_START >= start).to_numpy()
    if end is not None:
        ok &= (panel["date"] <= end).to_numpy()
    return np.where(ok)[0]

def cons_params(profile: np.ndarray) -> dict:
    risk, pat, agg = profile[:, 0], profile[:, 1], profile[:, 2]
    return {"w_ret": OBJ["w_return"], "w_dd": OBJ["w_drawdown"] * (1.5 - risk), "w_turn": OBJ["w_turnover"] * (0.5 + pat),
            "thr": OBJ["entry_threshold"] + 0.5 * (1.0 - agg), "soft_stop": 0.015 + 0.03 * risk, "vol_cap": 0.6 + 0.2 * risk}

def decision_scores(feat: dict):
    """edge (risk-adjusted trend evidence), risk (volatility percentile + drawdown depth), vol_pct. Decision-time only."""
    with np.errstate(invalid="ignore", divide="ignore"):
        sharpe60 = np.clip(feat["ret_60"] / (feat["rv_60"] * np.sqrt(60.0 / 252.0)), -3.0, 3.0)
        struct = 0.5 * np.sign(feat["sma50_sma200"]) + 0.5 * np.sign(feat["close_ema50"])
        edge = 0.6 * sharpe60 + 0.6 * struct
        vol_pct = 0.5 * (feat["atr_pct_pctile_252"] + feat["rv20_pctile_252"])
        risk = vol_pct + 0.5 * np.clip(-feat["dd_from_high_60"] / 0.10, 0.0, 2.0)
    return edge, risk, vol_pct

def conservative_rule_labels(feat, pos, profile, hold, pnl, port, allow_short=True):
    """Action label (0..3) from the conservative objective. Decision-time information only."""
    p = cons_params(profile); edge, risk, vol_pct = decision_scores(feat)
    pdd, gross, rloss = (np.asarray(port[k], dtype=float) for k in ("pdd", "gross", "rloss"))   # arrays, so ~ is a real boolean negation
    with np.errstate(invalid="ignore"):
        prohibited = ((vol_pct >= p["vol_cap"]) | (np.abs(feat["abnormal_ret_z"]) >= SOFT["abnormal_ret_z"])
                      | (feat["gap_abs_atr"] >= SOFT["gap_atr"]) | (feat["bench_dd_60"] <= -SOFT["market_stress_dd"]))
        portfolio_block = (pdd >= SOFT["portfolio_drawdown"]) | (gross >= SOFT["gross_exposure"]) | (rloss >= SOFT["recent_loss"])
        u_long = p["w_ret"] * edge - p["w_dd"] * risk - p["w_turn"]
        u_short = -p["w_ret"] * edge - p["w_dd"] * risk - p["w_turn"] - OBJ["short_penalty"]
        long_entry = (u_long >= p["thr"]) & ~prohibited & ~portfolio_block
        short_entry = (u_short >= p["thr"]) & ~prohibited & ~portfolio_block
        h_long, h_short = p["w_ret"] * edge - p["w_dd"] * risk, -p["w_ret"] * edge - p["w_dd"] * risk
        vol_spike = vol_pct >= SOFT["vol_exit_percentile"]
        derisk = (pdd >= SOFT["portfolio_drawdown"]) | ((rloss >= SOFT["recent_loss"]) & (pnl < 0))
        long_exit = ((pnl <= -p["soft_stop"]) | (h_long <= OBJ["exit_threshold"]) | vol_spike
                     | (feat["abnormal_ret_z"] <= -SOFT["abnormal_ret_z"]) | derisk)
        short_exit = ((pnl <= -p["soft_stop"]) | (h_short <= OBJ["exit_threshold"]) | vol_spike
                      | (feat["abnormal_ret_z"] >= SOFT["abnormal_ret_z"]) | derisk)
    y = np.zeros(len(pos), dtype=np.int64)
    flat = pos == 0
    y[flat & long_entry] = BUY
    if allow_short:
        y[flat & ~long_entry & short_entry] = SELL
    y[(pos == 1) & long_exit] = EXIT
    y[(pos == -1) & short_exit] = EXIT
    return y

def _finalize(rows, pos, prof, hold, pnl, port, y):
    valid = valid_action_mask(pos, CONFIG["allow_short"])
    assert valid[np.arange(len(y)), y].all(), "label contains an action that is invalid for its position"
    return {"row": np.asarray(rows, dtype=np.int64), "pos": np.asarray(pos, dtype=np.int64), "profile": np.asarray(prof, dtype=np.float32),
            "hold": np.asarray(hold), "pnl": np.asarray(pnl), "port": port, "y": np.asarray(y, dtype=np.int64), "valid": valid,
            "static": encode_static(pos, prof, pnl, hold, port["pdd"], port["gross"], port["size"], port["rloss"])}

def build_rule_samples(split: str, rng: np.random.Generator) -> dict:
    rows = eligible_rows(split)
    r = np.repeat(rows, 3); n = len(r)
    pos = np.tile(np.array([0, 1, -1]), len(rows))
    prof = rng.uniform(0.0, 1.0, size=(n, 3)).astype(np.float32)
    cap = np.minimum(CONFIG["hold_sample_max"], ROW_IN_SYM[r] + 1)
    hold = np.where(pos == 0, 0, 1 + np.floor(rng.random(n) * cap).astype(int))
    entry_px = OPEN_ARR[r - np.maximum(hold, 1) + 1]                      # hypothetical fill at the open `hold` bars ago
    pnl = np.where(pos == 0, 0.0, pos * (CLOSE_ARR[r] / entry_px - 1.0))
    # portfolio-level inputs cannot be derived from one symbol's history: sample broad, mostly-benign ranges
    size = np.where(pos == 0, 0.0, rng.uniform(0.02, 0.15, n))
    port = {"pdd": np.where(rng.random(n) < 0.55, rng.random(n) * 0.02, rng.random(n) * 0.15),
            "gross": np.maximum(rng.random(n) * 0.9, size),
            "size": size,
            "rloss": np.where(rng.random(n) < 0.6, 0.0, rng.random(n) * 0.08)}
    y = conservative_rule_labels({k: v[r] for k, v in RAW.items()}, pos, prof, hold, pnl, port, CONFIG["allow_short"])
    return _finalize(r, pos, prof, hold, pnl, port, y)

def build_trader_samples(split: str, csv_path: str) -> dict:
    """Behaviour cloning from genuine trader actions (optional). Required columns: symbol,date,action,position.
    Optional: risk_tolerance,patience,aggressiveness,unrealized_pnl,holding_bars,portfolio_drawdown,gross_exposure,position_size,recent_loss."""
    tl = pd.read_csv(csv_path)
    need = {"symbol", "date", "action", "position"}
    if not need <= set(tl.columns):
        raise ValueError(f"trader label file needs columns {sorted(need)}; has {list(tl.columns)}")
    tl["date"] = pd.to_datetime(tl["date"]).dt.normalize()
    names = {n: i for i, n in enumerate(ACTION_NAMES)}
    def _act(a):
        s = str(a).strip().upper()
        return names[s] if s in names else int(float(s))
    tl["action"] = tl["action"].map(_act)
    key = panel[["symbol", "date"]].assign(row=np.arange(len(panel)))
    m = tl.merge(key, on=["symbol", "date"], how="inner")
    m = m[m["row"].isin(eligible_rows(split))]
    pos = m["position"].to_numpy(dtype=int)
    if not np.isin(pos, [-1, 0, 1]).all() or not np.isin(m["action"], range(N_ACTIONS)).all():
        raise ValueError("trader labels contain invalid positions or actions")
    ok = valid_action_mask(pos, CONFIG["allow_short"])[np.arange(len(m)), m["action"].to_numpy(dtype=int)]
    if (~ok).any():
        print(f"  [{split}] dropping {(~ok).sum()} trader rows whose action is invalid for the given position")
    m, pos = m[ok], pos[ok]
    if len(m) == 0:
        raise ValueError(f"no usable trader labels in the {split} split")
    col = lambda c: m[c].to_numpy(dtype=float) if c in m.columns else np.zeros(len(m))
    prof = (m[PROFILE_FEATURES].to_numpy(dtype=np.float32) if set(PROFILE_FEATURES) <= set(m.columns)
            else np.tile(np.array(DEFAULT_PROFILE, dtype=np.float32), (len(m), 1)))
    port = {"pdd": col("portfolio_drawdown"), "gross": col("gross_exposure"), "size": col("position_size"), "rloss": col("recent_loss")}
    missing = [c for c in ("unrealized_pnl", "holding_bars", "portfolio_drawdown", "gross_exposure", "position_size", "recent_loss") if c not in m.columns]
    if missing:
        print(f"  [{split}] NOTE: columns not supplied, treated as 0: {missing}")
    return _finalize(m["row"].to_numpy(), pos, prof, col("holding_bars"), col("unrealized_pnl"), port, m["action"].to_numpy(dtype=int))

rng = np.random.default_rng(CONFIG["seed"])
if CONFIG["trader_labels_csv"]:
    LABEL_SOURCE = "trader_demonstrations"
    SAMPLES = {s: build_trader_samples(s, CONFIG["trader_labels_csv"]) for s in ("train", "val", "test")}
else:
    LABEL_SOURCE = "strategy_derived_proxy (conservative objective, decision-time)"
    SAMPLES = {s: build_rule_samples(s, rng) for s in ("train", "val", "test")}

print("=" * 96)
if LABEL_SOURCE.startswith("strategy_derived_proxy"):
    print("LABEL SOURCE: STRATEGY-DERIVED PROXY LABELS from an explicit conservative objective. NOT human demonstrations.")
    print("Decision-time information only: no future price is used for any label.")
else:
    print("LABEL SOURCE: genuine trader demonstrations (behaviour cloning).")
print("=" * 96)
dist = pd.DataFrame({s: np.bincount(S["y"], minlength=N_ACTIONS) for s, S in SAMPLES.items()}, index=ACTION_NAMES)
print(dist.assign(**{f"{c}_%": (100 * dist[c] / dist[c].sum()).round(2) for c in dist.columns}))
for s, S in SAMPLES.items():
    r = panel.loc[S["row"], "date"]
    print(f"{s:5s}: {len(S['y']):>9,} samples | decision dates {r.min().date()} -> {r.max().date()}")
assert panel.loc[SAMPLES["train"]["row"], "date"].max() < panel.loc[SAMPLES["val"]["row"], "date"].min()
assert panel.loc[SAMPLES["val"]["row"], "date"].max() < panel.loc[SAMPLES["test"]["row"], "date"].min()
print("chronology check passed: train < validation < test (no overlap of decision dates)")

if LABEL_SOURCE.startswith("strategy_derived_proxy"):
    _tr = SAMPLES["train"]; _fl = _tr["pos"] == 0
    _bal = np.tile(np.array(DEFAULT_PROFILE, dtype=np.float32), (len(_tr["row"]), 1))
    _e, _rk, _vp = decision_scores({k: v[_tr["row"]] for k, v in RAW.items()})
    print("\nflat-state labels in TRAIN: share of BUY/SELL (entries) vs HOLD by volatility percentile bucket (proves entries avoid high volatility):")
    _b = pd.cut(pd.Series(_vp[_fl]), [0, 0.3, 0.6, 0.8, 1.0], include_lowest=True)
    print(pd.crosstab(_b, pd.Series(_tr["y"][_fl]).map(dict(enumerate(ACTION_NAMES))), normalize="index").round(3).to_string())

_cnt = np.bincount(SAMPLES["train"]["y"], minlength=N_ACTIONS).astype(float)
_w = np.where(_cnt > 0, _cnt.sum() / (N_ACTIONS * np.maximum(_cnt, 1)), 0.0)
_w = np.minimum(_w, CONFIG["train"]["max_class_weight"])
CLASS_WEIGHTS = (_w * (_cnt.sum() / (_w * _cnt).sum())).astype(np.float32)
print("\nclass weights:", {n: round(float(w), 3) for n, w in zip(ACTION_NAMES, CLASS_WEIGHTS)})
if (_cnt == 0).any():
    print("NOTE: classes with no training examples:", [ACTION_NAMES[i] for i in np.where(_cnt == 0)[0]])

_fit = (panel["date"] <= D_TRAIN_END).to_numpy()
_med = np.nanmedian(RAW_MAT[_fit], axis=0)
_imp = np.where(np.isfinite(RAW_MAT[_fit]), RAW_MAT[_fit], _med)
_std = _imp.std(axis=0); _std = np.where(_std < 1e-8, 1.0, _std)
PREPROCESSING = {"feature_columns": list(FEATURE_COLUMNS), "median": _med.astype(np.float32),
                 "mean": _imp.mean(axis=0).astype(np.float32), "std": _std.astype(np.float32),
                 "clip": float(CONFIG["train"]["scaler_clip"]),
                 "fit_rows_end_date": str(D_TRAIN_END.date()), "n_fit_rows": int(_fit.sum())}
X_SCALED = scale_features(RAW_MAT, PREPROCESSING["median"], PREPROCESSING["mean"], PREPROCESSING["std"], PREPROCESSING["clip"])
print(f"scaler fit on {PREPROCESSING['n_fit_rows']:,} rows with dates <= {PREPROCESSING['fit_rows_end_date']}; "
      f"non-finite feature cells imputed overall: {int((~np.isfinite(RAW_MAT)).sum()):,}")

## 6. Model definition
The network lives in `sp05_model.py` (written by the next cell, shipped with the artifacts); it contains **no data-download or training code** and no safety logic.

* `ConservativePolicyNet`: GRU or LSTM encoder over the last `seq_len` bars (43 input columns) → last hidden state, concatenated with a dense branch for the 12-dim static input → dense layer → 4 action logits. `seq_len`, `hidden_size`, `num_layers`, `encoder` live in `CONFIG`.
* `ConservativePolicy`: `predict(state)`, `save(path)`, `load(path, device)`. It scales raw features with the saved preprocessing, masks impossible actions by position and returns **temperature-calibrated** probabilities. Loading needs only the saved file: no downloads, no training, `weights_only=True`, `eval()` mode, so predictions are deterministic.
* **The returned action is a proposal.** Hard constraints are enforced by `RiskController` in `sp05_risk.py`, never by the network.

**Confidence and its limits** (please read): the confidence is the temperature-scaled probability of the chosen action among the *valid* actions. The temperature is fitted on the validation split against the *proxy labels*, so confidence means "how strongly the network agrees with the label rule", not "probability that this action avoids a loss". Calibration was fitted on one period and may drift; it is not calibrated for rare high-stress states, which are exactly where it matters most; it is not comparable with the other policies' confidences. The hard gate therefore never relies on it, except for one optional minimum-confidence entry filter that can only make the system *more* conservative.

**`predict(state)` contract** (same keys as the other policies):
`{"sequence": (L, 30) shared RAW features, "risk_features": (L, 13) from RiskFeatureAdapter, "position": -1|0|+1, "profile": [risk, patience, aggressiveness], "unrealized_pnl", "holding_bars", "portfolio_drawdown", "gross_exposure", "position_size", "recent_loss"}`; the last six are optional (default 0). A single pre-extended `(L, 43)` sequence is accepted as well.

In [ ]:
%%writefile sp05_model.py
"""SP-05 Conservative Risk-Aware Policy - network and inference wrapper.

The policy only PROPOSES actions. A separate deterministic gate (sp05_risk.RiskController) decides what may execute.

Contains NO data download and NO training code. Load a saved policy with
``ConservativePolicy.load(dir_or_pt_file, device)`` and call ``predict(state)``.
"""
from __future__ import annotations

import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sp05_features import (
    ACTION_NAMES, DEFAULT_PROFILE, FEATURE_COLUMNS, N_ACTIONS, N_BASE_FEATURES, N_FEATURES, N_STATIC, RiskFeatureAdapter,
    PROFILE_FEATURES, WARMUP_BARS, build_feature_schema, build_features, encode_static,
    scale_features, valid_action_mask,
)

ARTIFACT_NAME = "conservative_policy.pt"
CONFIG_NAME = "conservative_config.json"
MASK_VALUE = -1e9


class ConservativePolicyNet(nn.Module):
    """GRU/LSTM encoder over market features + static (position, profile) branch + action head."""

    def __init__(self, n_features: int = N_FEATURES, n_static: int = N_STATIC, hidden_size: int = 64,
                 num_layers: int = 2, dropout: float = 0.25, static_hidden: int = 16,
                 head_hidden: int = 64, encoder: str = "gru", n_actions: int = N_ACTIONS):
        super().__init__()
        if encoder not in ("gru", "lstm"):
            raise ValueError("encoder must be 'gru' or 'lstm'")
        rnn_cls = nn.GRU if encoder == "gru" else nn.LSTM
        self.rnn = rnn_cls(n_features, hidden_size, num_layers=num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0.0)
        self.static = nn.Sequential(nn.Linear(n_static, static_hidden), nn.GELU())
        self.head = nn.Sequential(
            nn.Linear(hidden_size + static_hidden, head_hidden), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(head_hidden, n_actions),
        )

    def forward(self, seq: torch.Tensor, static: torch.Tensor) -> torch.Tensor:
        out, _ = self.rnn(seq)
        z = torch.cat([out[:, -1, :], self.static(static)], dim=1)
        return self.head(z)


def mask_logits(logits: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
    return logits.masked_fill(~valid, MASK_VALUE)


def _jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    return obj


class ConservativePolicy:
    """Inference wrapper: scaling + network + action masking + calibrated confidence."""

    def __init__(self, net: ConservativePolicyNet, config: dict, preprocessing: dict, device="cpu"):
        self.device = torch.device(device)
        self.net = net.to(self.device).eval()
        self.config = _jsonable(config)
        self.pre = {
            "feature_columns": list(preprocessing["feature_columns"]),
            "median": np.asarray(preprocessing["median"], dtype=np.float32),
            "mean": np.asarray(preprocessing["mean"], dtype=np.float32),
            "std": np.asarray(preprocessing["std"], dtype=np.float32),
            "clip": float(preprocessing["clip"]),
        }
        if self.pre["feature_columns"] != list(FEATURE_COLUMNS):
            raise ValueError("feature order in preprocessing does not match sp05_features.FEATURE_COLUMNS")
        self.seq_len = int(self.config["seq_len"])
        self.allow_short = bool(self.config.get("allow_short", True))
        self.temperature = float(self.config.get("temperature", 1.0))

    # ------------------------------------------------------------------ scaling
    def scale(self, seq: np.ndarray) -> np.ndarray:
        """Raw (..., F) -> scaled float32. NaN -> train median, standardise, clip."""
        return scale_features(seq, self.pre["median"], self.pre["mean"], self.pre["std"], self.pre["clip"])

    # ------------------------------------------------------------------ inference
    @torch.no_grad()
    def predict_scaled(self, seq_scaled: torch.Tensor, static: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
        """Calibrated probabilities (B, 4) from already-scaled tensors on self.device."""
        logits = mask_logits(self.net(seq_scaled, static), valid)
        return torch.softmax(logits / self.temperature, dim=1)

    @torch.no_grad()
    def predict(self, state: dict) -> dict:
        """Predict an action for one state or a batch of states.

        state keys
        ----------
        sequence : array (L, 30) or (B, L, 30) of RAW shared features (FEATURE_COLUMNS[:30]), oldest bar first,
                   plus 'risk_features' (L, 13) / (B, L, 13) from RiskFeatureAdapter; OR a single
                   (L, 43) / (B, L, 43) array already extended with the risk columns.
                   L >= seq_len; the last seq_len bars are used.
        position : int in {-1, 0, 1} (or length-B array). Default 0 (flat).
        profile  : dict or length-3 / (B, 3) array [risk_tolerance, patience, aggressiveness]
                   in [0, 1]. Default (0.5, 0.5, 0.5).
        unrealized_pnl : (optional) position-signed fractional P&L since entry, scalar or per-state. Default 0.
        holding_bars   : (optional) bars held, entry bar = 1, scalar or per-state. Default 0.
                   Both are ignored (zeroed) when position == 0.
        portfolio_drawdown : (optional) fraction the portfolio equity is below its peak, >= 0. Default 0.
        gross_exposure     : (optional) portfolio gross notional / equity. Default 0.
        position_size      : (optional) this position's notional / equity (ignored when flat). Default 0.
        recent_loss        : (optional) fall of portfolio equity over ~20 bars as a fraction, >= 0. Default 0.

        The returned action is only a PROPOSAL. Hard safety constraints are enforced by sp05_risk.RiskController,
        never by this network, and the controller always has final authority.

        Returns a dict with action_probs, action, action_name, confidence, valid_actions, entropy.
        Single-state input yields scalars / 1-D arrays; batch input yields arrays.
        """
        seq = np.asarray(state["sequence"], dtype=np.float32)
        single = seq.ndim == 2
        if single:
            seq = seq[None]
        if seq.ndim != 3:
            raise ValueError(f"sequence must have shape (L, F) or (B, L, F); got {seq.shape}")
        if seq.shape[2] == N_BASE_FEATURES:                      # shared 30-column contract + adapter output
            extra = state.get("risk_features")
            if extra is None:
                raise ValueError(f"sequence has the shared {N_BASE_FEATURES} columns, so 'risk_features' (L, "
                                 f"{N_FEATURES - N_BASE_FEATURES}) is required: build it with "
                                 "RiskFeatureAdapter().adapt_state(state, ohlcv) or policy.state_from_ohlcv(...)")
            extra = np.asarray(extra, dtype=np.float32)
            if extra.ndim == 2:
                extra = extra[None]
            if extra.shape[0] != seq.shape[0] or extra.shape[1] != seq.shape[1] or extra.shape[2] != N_FEATURES - N_BASE_FEATURES:
                raise ValueError(f"risk_features must have shape (L, {N_FEATURES - N_BASE_FEATURES}) or (B, L, "
                                 f"{N_FEATURES - N_BASE_FEATURES}) aligned with sequence {seq.shape[:2]}; got {extra.shape}")
            seq = np.concatenate([seq, extra], axis=2)
        elif seq.shape[2] != N_FEATURES:
            raise ValueError(f"sequence must have {N_BASE_FEATURES} (shared) or {N_FEATURES} (extended) columns; got {seq.shape[2]}")
        if seq.shape[1] < self.seq_len:
            raise ValueError(f"need at least seq_len={self.seq_len} bars; got {seq.shape[1]}")
        seq = seq[:, -self.seq_len:, :]
        batch = seq.shape[0]

        position = np.asarray(state.get("position", 0)).astype(int).reshape(-1)
        if position.size == 1 and batch > 1:
            position = np.repeat(position, batch)
        if position.size != batch:
            raise ValueError("position must be scalar or have one entry per sequence")
        static = encode_static(position, state.get("profile", DEFAULT_PROFILE),
                               state.get("unrealized_pnl"), state.get("holding_bars"), state.get("portfolio_drawdown"),
                               state.get("gross_exposure"), state.get("position_size"), state.get("recent_loss"))
        valid = valid_action_mask(position, self.allow_short)

        seq_t = torch.from_numpy(self.scale(seq)).to(self.device)
        static_t = torch.from_numpy(static).to(self.device)
        valid_t = torch.from_numpy(valid).to(self.device)
        probs = self.predict_scaled(seq_t, static_t, valid_t).cpu().numpy()

        action = probs.argmax(axis=1)
        confidence = probs[np.arange(batch), action]
        entropy = -(probs * np.log(np.clip(probs, 1e-12, 1.0))).sum(axis=1)
        names = [ACTION_NAMES[a] for a in action]
        if single:
            return {"action_probs": probs[0], "action": int(action[0]), "action_name": names[0],
                    "confidence": float(confidence[0]), "valid_actions": valid[0], "entropy": float(entropy[0])}
        return {"action_probs": probs, "action": action, "action_name": names,
                "confidence": confidence, "valid_actions": valid, "entropy": entropy}

    def state_from_ohlcv(self, ohlcv: pd.DataFrame, benchmark: pd.DataFrame, position: int = 0, profile=None,
                         unrealized_pnl: float = 0.0, holding_bars: int = 0, portfolio_drawdown: float = 0.0,
                         gross_exposure: float = 0.0, position_size: float = 0.0, recent_loss: float = 0.0) -> dict:
        """Build a predict() state from OHLCV frames the caller already holds (no downloading)."""
        feats = build_features(ohlcv, benchmark)[FEATURE_COLUMNS]
        if len(feats) < WARMUP_BARS + self.seq_len:
            raise ValueError(f"need at least {WARMUP_BARS + self.seq_len} bars of history; got {len(feats)}")
        window = feats.iloc[-self.seq_len:]
        base = window.to_numpy(dtype=np.float32)[:, :N_BASE_FEATURES]            # shared 30-column sequence
        extra = window.to_numpy(dtype=np.float32)[:, N_BASE_FEATURES:]           # same values the adapter produces
        return {"sequence": base, "risk_features": extra, "position": int(position),
                "profile": profile if profile is not None else DEFAULT_PROFILE,
                "unrealized_pnl": unrealized_pnl, "holding_bars": holding_bars, "portfolio_drawdown": portfolio_drawdown,
                "gross_exposure": gross_exposure, "position_size": position_size, "recent_loss": recent_loss}

    # ------------------------------------------------------------------ persistence
    def save(self, path) -> Path:
        """Write conservative_policy.pt, conservative_config.json, feature_schema.json, preprocessing.pkl into `path`.

        conservative_policy.pt is self-contained and authoritative; the other three are standard
        side-car copies for tooling and the Meta-RL selector.
        """
        out = Path(path)
        out.mkdir(parents=True, exist_ok=True)
        schema = build_feature_schema(self.seq_len)
        pre_plain = {"feature_columns": self.pre["feature_columns"], "median": self.pre["median"].tolist(),
                     "mean": self.pre["mean"].tolist(), "std": self.pre["std"].tolist(), "clip": self.pre["clip"]}
        config = dict(self.config)
        config["temperature"] = self.temperature
        bundle = {
            "format_version": 1,
            "state_dict": {k: v.detach().cpu() for k, v in self.net.state_dict().items()},
            "config": config,
            "preprocessing": pre_plain,
            "feature_schema": schema,
        }
        torch.save(bundle, out / ARTIFACT_NAME)
        (out / CONFIG_NAME).write_text(json.dumps(_jsonable(config), indent=2))
        (out / "feature_schema.json").write_text(json.dumps(_jsonable(schema), indent=2))
        with open(out / "preprocessing.pkl", "wb") as fh:
            pickle.dump({**pre_plain, "median": self.pre["median"], "mean": self.pre["mean"], "std": self.pre["std"]}, fh)
        return out / ARTIFACT_NAME

    @classmethod
    def load(cls, path, device=None) -> "ConservativePolicy":
        """Load from a directory containing conservative_policy.pt, or from the .pt file itself."""
        p = Path(path)
        pt = p / ARTIFACT_NAME if p.is_dir() else p
        if not pt.exists():
            raise FileNotFoundError(f"policy file not found: {pt}")
        device = torch.device(device) if device is not None else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        bundle = torch.load(pt, map_location=device, weights_only=True)
        config = bundle["config"]
        net = ConservativePolicyNet(**config["model"])
        net.load_state_dict(bundle["state_dict"])
        return cls(net, config, bundle["preprocessing"], device)


In [ ]:
import sp05_model
importlib.reload(sp05_model)
from sp05_model import ConservativePolicyNet, ConservativePolicy, mask_logits

set_seed(CONFIG["seed"])
MODEL_CFG = {"n_features": N_FEATURES, "n_static": N_STATIC, "n_actions": N_ACTIONS, **CONFIG["model"]}
net = ConservativePolicyNet(**MODEL_CFG).to(DEVICE)
print(net)
print(f"trainable parameters: {sum(p.numel() for p in net.parameters() if p.requires_grad):,}")

## 7. Training
Supervised classification of the (proxy or genuine) action labels with a class-weighted cross-entropy on **position-masked logits**. Early stopping uses the validation loss; the best weights are restored. Windows are gathered on the fly from the scaled feature matrix (no giant window tensor is stored).

In [ ]:
X_ALL = torch.from_numpy(X_SCALED).to(DEVICE)
OFFSETS = torch.arange(-(L - 1), 1, device=DEVICE)
OFFSETS_NP = np.arange(-(L - 1), 1)

def to_torch(S):
    return {"row": torch.from_numpy(S["row"]).long().to(DEVICE), "static": torch.from_numpy(S["static"]).to(DEVICE),
            "y": torch.from_numpy(S["y"]).long().to(DEVICE), "valid": torch.from_numpy(S["valid"]).to(DEVICE)}

TS = {k: to_torch(v) for k, v in SAMPLES.items()}

def get_batch(T, idx):
    win = T["row"][idx].unsqueeze(1) + OFFSETS            # (B, L) global row ids, all inside one symbol block
    return X_ALL[win], T["static"][idx], T["valid"][idx], T["y"][idx]

@torch.no_grad()
def forward_all(model, T, batch_size=4096):
    """Masked logits for every sample of a split (CPU tensor)."""
    model.eval(); out = []
    for i in range(0, len(T["y"]), batch_size):
        idx = torch.arange(i, min(i + batch_size, len(T["y"])), device=DEVICE)
        seq, st, valid, _ = get_batch(T, idx)
        out.append(mask_logits(model(seq, st), valid).cpu())
    return torch.cat(out)

tc = CONFIG["train"]
class_w = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32, device=DEVICE)
optimizer = torch.optim.AdamW(net.parameters(), lr=tc["lr"], weight_decay=tc["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)
y_val_cpu = TS["val"]["y"].cpu()
gen = torch.Generator().manual_seed(CONFIG["seed"])
n_train = len(TS["train"]["y"])

history, best_val, best_state, bad_epochs = [], float("inf"), None, 0
t0 = time.time()
for epoch in range(1, tc["epochs"] + 1):
    net.train()
    perm = torch.randperm(n_train, generator=gen).to(DEVICE)       # shuffling happens inside the TRAIN split only
    total, seen = 0.0, 0
    for i in range(0, n_train, tc["batch_size"]):
        idx = perm[i:i + tc["batch_size"]]
        seq, st, valid, y = get_batch(TS["train"], idx)
        if tc["input_noise_std"] > 0:
            seq = seq + tc["input_noise_std"] * torch.randn_like(seq)
        loss = Fnn.cross_entropy(mask_logits(net(seq, st), valid), y, weight=class_w)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), tc["grad_clip"])
        optimizer.step()
        total += loss.item() * len(idx); seen += len(idx)
    train_loss = total / seen
    val_logits = forward_all(net, TS["val"])
    val_loss = Fnn.cross_entropy(val_logits, y_val_cpu, weight=class_w.cpu()).item()
    val_f1 = f1_score(y_val_cpu.numpy(), val_logits.argmax(1).numpy(), average="macro", labels=list(range(N_ACTIONS)), zero_division=0)
    scheduler.step(val_loss)
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_macro_f1": val_f1,
                    "lr": optimizer.param_groups[0]["lr"]})
    improved = val_loss < best_val - 1e-5
    if improved:
        best_val, bad_epochs = val_loss, 0
        best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
    else:
        bad_epochs += 1
    print(f"epoch {epoch:3d} | train {train_loss:.4f} | val {val_loss:.4f} | val macro-F1 {val_f1:.3f} | lr {optimizer.param_groups[0]['lr']:.1e}"
          f"{'  *' if improved else ''}  [{time.time() - t0:.0f}s]")
    if bad_epochs >= tc["patience"]:
        print(f"early stopping at epoch {epoch} (no validation improvement for {tc['patience']} epochs)")
        break

net.load_state_dict(best_state); net.eval()
history_df = pd.DataFrame(history)
BEST_EPOCH = int(history_df.loc[history_df["val_loss"].idxmin(), "epoch"])
print(f"\nbest epoch: {BEST_EPOCH} | best validation loss: {best_val:.4f}")

## 8. Calibration and classification metrics
* **Temperature scaling** is fit on validation logits only and then frozen.
* Metrics on the chronological **validation** and **test** splits: accuracy, balanced accuracy, macro/weighted F1, per-class precision/recall/F1, confusion matrix, NLL and ECE. They measure **imitation of the proxy labels** (unless you supplied trader actions), not safety and not profitability.
* **By risk condition:** the same predictions are split into calm flat states, flat states where entries are *prohibited* by the soft rules, flat states where *portfolio limits* are breached, and held positions with elevated or normal risk. The key question is whether the network proposes entries where the rules forbid them.

In [ ]:
def fit_temperature(logits: torch.Tensor, y: torch.Tensor) -> float:
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)
    def closure():
        opt.zero_grad()
        loss = Fnn.cross_entropy(logits / torch.exp(log_t), y)
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.exp(log_t).clamp(0.05, 10.0).item())

def ece_score(conf, correct, bins=10):
    edges = np.linspace(0, 1, bins + 1); ece, rows = 0.0, []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
            rows.append({"bin_lo": lo, "bin_hi": hi, "n": int(m.sum()), "confidence": float(conf[m].mean()), "accuracy": float(correct[m].mean())})
    return float(ece), rows

def classification_summary(logits, y, temperature):
    y = np.asarray(y); probs = torch.softmax(logits / temperature, dim=1).numpy()
    pred = probs.argmax(1); conf = probs.max(1); labels = list(range(N_ACTIONS))
    p, r, f, s = precision_recall_fscore_support(y, pred, labels=labels, zero_division=0)
    ece, bins = ece_score(conf, (pred == y).astype(float))
    return {"n_samples": int(len(y)), "accuracy": float(accuracy_score(y, pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
            "macro_f1": float(f1_score(y, pred, average="macro", labels=labels, zero_division=0)),
            "weighted_f1": float(f1_score(y, pred, average="weighted", labels=labels, zero_division=0)),
            "majority_class_baseline_accuracy": float(np.bincount(y, minlength=N_ACTIONS).max() / len(y)),
            "nll": float(-np.log(np.clip(probs[np.arange(len(y)), y], 1e-12, 1.0)).mean()),
            "ece": ece, "reliability_bins": bins, "temperature": float(temperature),
            "per_class": {ACTION_NAMES[i]: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]), "support": int(s[i])} for i in labels},
            "confusion_matrix": confusion_matrix(y, pred, labels=labels).tolist(), "pred": pred}

LOGITS = {k: forward_all(net, TS[k]) for k in ("val", "test")}
Y = {k: TS[k]["y"].cpu().numpy() for k in ("val", "test")}
TEMPERATURE = fit_temperature(LOGITS["val"], TS["val"]["y"].cpu())
print(f"fitted temperature (validation only): {TEMPERATURE:.3f}")

CLS = {k: classification_summary(LOGITS[k], Y[k], TEMPERATURE) for k in ("val", "test")}
CLS_UNCAL_TEST = classification_summary(LOGITS["test"], Y["test"], 1.0)
for k in ("val", "test"):
    c = CLS[k]
    print(f"\n[{k}] acc {c['accuracy']:.3f} | balanced acc {c['balanced_accuracy']:.3f} | macro-F1 {c['macro_f1']:.3f} | "
          f"weighted-F1 {c['weighted_f1']:.3f} | majority baseline {c['majority_class_baseline_accuracy']:.3f}")
    print(f"      NLL {c['nll']:.4f} | ECE {c['ece']:.4f}" + (f"  (uncalibrated test: NLL {CLS_UNCAL_TEST['nll']:.4f}, ECE {CLS_UNCAL_TEST['ece']:.4f})" if k == "test" else ""))
    print(pd.DataFrame(c["per_class"]).T.round(3).to_string())
    print("confusion matrix (rows = true, cols = predicted):")
    print(pd.DataFrame(c["confusion_matrix"], index=ACTION_NAMES, columns=ACTION_NAMES).to_string())


policy_config = {
    "model_name": "SP-05 Conservative Risk-Aware Policy", "policy_id": "SP-05", "version": "1.0.0", "artifact": "conservative_policy.pt",
    "seq_len": L, "allow_short": CONFIG["allow_short"], "temperature": TEMPERATURE,
    "model": MODEL_CFG, "action_space": {str(i): n for i, n in enumerate(ACTION_NAMES)},
    "label_source": LABEL_SOURCE,
    "label_note": ("Labels are strategy-derived PROXY labels from an explicit conservative objective: NOT human demonstrations."
                   if LABEL_SOURCE.startswith("strategy_derived_proxy") else "Labels are genuine trader demonstrations (behaviour cloning)."),
    "label_definition": {"information": "decision-time only (features, position, P&L, holding, portfolio state, profile)", "future_prices_used": False,
                         "objective": CONFIG["objective"], "soft_limits": CONFIG["soft_limits"]},
    "confidence_definition": "temperature-calibrated probability of the chosen action after masking invalid actions",
    "confidence_limitations": ["calibrated against proxy labels on the validation period, not against loss avoidance",
                               "not validated for rare high-stress states", "not comparable across policies",
                               "the hard risk gate does not depend on it (except an optional minimum-confidence entry filter that only adds caution)"],
    "authority": "This policy only PROPOSES actions. sp05_risk.RiskController has final authority over execution.",
    "risk_limits": RISK_LIMITS.to_dict(),
    "execution_convention": "decision at close of bar t, fill at open of bar t+1",
    "shared_schema": {"n_shared_sequence_features": N_BASE_FEATURES, "n_risk_features": len(EXTRA_FEATURE_COLUMNS),
                      "n_shared_static_features": N_SHARED_STATIC, "n_portfolio_static_extras": N_STATIC - N_SHARED_STATIC,
                      "adapter": "sp05_features.RiskFeatureAdapter"},
    "data": {"symbols": list(raw_data), "benchmark": CONFIG["benchmark"], "start_date": CONFIG["start_date"], "end_date": CONFIG["end_date"],
             "train_end": CONFIG["train_end"], "val_end": CONFIG["val_end"], "source": "yfinance daily adjusted OHLCV"},
    "training": {**CONFIG["train"], "seed": CONFIG["seed"], "best_epoch": BEST_EPOCH, "class_weights": CLASS_WEIGHTS.tolist(),
                 "n_train_samples": int(n_train)},
    "intended_use": "research and paper trading only; not a recommendation to trade real money; no capital protection is claimed",
}
policy = ConservativePolicy(net, policy_config, PREPROCESSING, DEVICE)
print("\nConservativePolicy ready (in-memory).")


In [ ]:
# ---- classification quality by risk condition (validation and test) ---------------------------------------------
COND_NAMES = {0: "flat: calm, portfolio within soft limits", 1: "flat: entry PROHIBITED (volatility / abnormal move / market stress)",
              2: "flat: portfolio soft limit breached (drawdown / exposure / recent loss)",
              3: "holding: risk elevated (vol spike / de-risk trigger)", 4: "holding: calm"}

def condition_codes(S):
    r, pos, prof = S["row"], S["pos"], S["profile"]
    feat = {k: v[r] for k, v in RAW.items()}
    p = cons_params(prof); edge, risk, vol_pct = decision_scores(feat); port = S["port"]
    with np.errstate(invalid="ignore"):
        prohibited = ((vol_pct >= p["vol_cap"]) | (np.abs(feat["abnormal_ret_z"]) >= SOFT["abnormal_ret_z"])
                      | (feat["gap_abs_atr"] >= SOFT["gap_atr"]) | (feat["bench_dd_60"] <= -SOFT["market_stress_dd"]))
        pblock = (port["pdd"] >= SOFT["portfolio_drawdown"]) | (port["gross"] >= SOFT["gross_exposure"]) | (port["rloss"] >= SOFT["recent_loss"])
        elevated = (vol_pct >= SOFT["vol_exit_percentile"]) | (port["pdd"] >= SOFT["portfolio_drawdown"])
    cond = np.zeros(len(r), dtype=int)
    cond[(pos == 0) & pblock] = 2
    cond[(pos == 0) & prohibited] = 1
    cond[(pos != 0) & ~elevated] = 4
    cond[(pos != 0) & elevated] = 3
    return cond

COND_CLS, VIOLATIONS = {}, {}
for split in ("val", "test"):
    S = SAMPLES[split]; y = Y[split]; pred = CLS[split]["pred"]; cond = condition_codes(S); rows_ = []
    for cd_, name in COND_NAMES.items():
        m = cond == cd_
        if m.sum() == 0:
            continue
        present = np.unique(np.concatenate([y[m], pred[m]]))
        rows_.append({"condition": name, "n": int(m.sum()), "accuracy": float((y[m] == pred[m]).mean()),
                      "macro_f1_present": float(f1_score(y[m], pred[m], average="macro", labels=present, zero_division=0)),
                      **{f"true_{a}": int((y[m] == i).sum()) for i, a in enumerate(ACTION_NAMES)},
                      **{f"pred_{a}": int((pred[m] == i).sum()) for i, a in enumerate(ACTION_NAMES)}})
    COND_CLS[split] = pd.DataFrame(rows_).set_index("condition")
    forb = np.isin(cond, [1, 2])
    VIOLATIONS[split] = {"flat_forbidden_states": int(forb.sum()),
                         "network_proposes_entry_in_forbidden_states_%": float(100 * np.isin(pred[forb], [BUY, SELL]).mean()) if forb.any() else float("nan"),
                         "labels_with_entry_in_forbidden_states_%": float(100 * np.isin(y[forb], [BUY, SELL]).mean()) if forb.any() else float("nan")}
    print(f"\n[{split}] classification by risk condition:")
    print(COND_CLS[split].round(3).to_string())
    print(f"[{split}] entry proposals in states where the soft rules forbid entries: {VIOLATIONS[split]}")
print("\nAny non-zero violation rate above is a case where the NETWORK proposed something the rule forbids; the hard gate exists for exactly that.")

## 9. Backtest with and without the risk gate, stress tests and failure analysis
A portfolio simulator with explicit cash, shares and constraints. In every run the policy (or baseline) *proposes* an action for each symbol and the deterministic gate (when enabled) *decides* what executes.

* **Timing:** decisions use bar *t* (close known); orders fill at the **open of bar t+1**. Fees, slippage (every fill is moved against us) and short borrow are charged.
* **Capital:** an entry targets `min(position_fraction, max_position_size) × equity`, multiplied by a confidence factor when `confidence_sizing` is on (smaller when less sure), limited by available cash and exposure caps. Exits run first; competing entries are ranked by confidence. The policy is fed the **real** portfolio state each day: drawdown below the **trailing 120-bar equity high** (a rolling high-water mark: with an all-time peak, an account that went to cash after a loss could never recover to the old peak and would be locked out of new entries forever), gross exposure, this position's size and the equity fall over ~20 bars.
* **Runs reported:** `SP-05 policy + gate` (the intended system; balanced / conservative / aggressive profiles and a long-only variant), `SP-05 policy alone` (no gate; shows what the network does by itself), `Always-long + gate` (the gate with a naive policy; shows what the gate does by itself), a **simple rule-based risk controller** (long-only: own trend/volatility/drawdown rules, run through the same gate), and the decision-time label rule itself. Benchmarks: SPY and equal-weight universe buy-and-hold (no costs, no gate), plus **SPY scaled to the main run's average exposure** (the rest in cash), because a system that is mostly out of the market looks "safe" simply by holding little; compare drawdown and Sharpe against that row, not only against fully invested buy-and-hold.
* **Gate accounting:** every override is logged by reason (blocked entries, forced exits). For blocked entries the later forward return is shown as *hindsight evaluation only*, so you can see what the gate saved *and* what it cost.
* **Stress tests** (section 9c) use past-only definitions of high-volatility and severe-drawdown periods on the benchmark, plus arithmetic gap-loss budgets, and **failures** (9d) list the worst days, drawdowns and forced stop-outs.

The test split is untouched by training, calibration and early stopping, but it is one regime and a short window. No result here is evidence of future capital protection.

In [ ]:
PROFILES = {"balanced": (0.5, 0.5, 0.5), "conservative": (0.2, 0.8, 0.2), "aggressive": (0.8, 0.3, 0.8)}
FEE, SLIP = CONFIG["fee_bps"] / 1e4, CONFIG["slippage_bps"] / 1e4
BORROW_DAILY = CONFIG["short_borrow_bps_annual"] / 1e4 / 252.0
CAP0 = float(CONFIG["initial_capital"])
OFFSETS_NP = np.arange(-(L - 1), 1)

def prepare_backtest(split: str) -> dict:
    rows = eligible_rows(split)                                         # no outcome needed: backtests never use labels
    sub = panel.loc[rows, ["symbol", "date"]].assign(row=rows)
    piv = sub.pivot(index="date", columns="symbol", values="row")
    syms = list(piv.columns)
    n_all = len(piv); piv = piv.dropna(how="any")
    if len(piv) < n_all:
        print(f"  [{split}] {n_all - len(piv)} dates dropped because not every symbol had a complete window")
    rows_mat = piv[syms].to_numpy().astype(np.int64)
    bclose = bench_df["Close"].reindex(piv.index).ffill(limit=3).to_numpy(dtype=float)
    if np.isnan(bclose).any():
        raise RuntimeError("benchmark close has gaps inside the backtest window")
    return {"split": split, "dates": piv.index, "symbols": syms, "rows": rows_mat,
            "open": OPEN_ARR[rows_mat], "close": CLOSE_ARR[rows_mat], "bench_close": bclose}


GATE = RiskController(RISK_LIMITS)
DD_LOOKBACK = int(CONFIG["dd_lookback_bars"])

def run_portfolio_backtest(bt: dict, decide_fn, allow_short: bool, gate=None, conf_sizing: bool = False) -> dict:
    T, S = bt["open"].shape
    cash = CAP0; shares = np.zeros(S); entry_px = np.zeros(S); entry_k = np.zeros(S, dtype=int); entry_conf = np.zeros(S); hold = np.zeros(S, dtype=int)
    eq_hist = [CAP0]
    ret_hist, eq_out, expo_hist, turn_hist, pdd_hist, trades, events = [], [], [], [], [], [], []
    reason_counts = np.zeros(len(REASON_NAMES), dtype=int); override_days = []
    n_entries = n_skipped = 0
    cap_gross = CONFIG["max_gross_exposure"] if gate is None else min(CONFIG["max_gross_exposure"], gate.limits.max_gross_exposure)
    entry_frac = min(CONFIG["position_fraction"], CONFIG["max_position_size"])
    if gate is not None:
        entry_frac = min(entry_frac, gate.limits.max_position_fraction)
    for k in range(T - 1):
        prev_eq = eq_hist[-1]; peak = max(eq_hist[-(DD_LOOKBACK + 1):])               # trailing high-water mark (not the all-time peak)
        pdd = max(0.0, 1.0 - prev_eq / peak)
        values = shares * bt["close"][k]
        gross_now = float(np.abs(values).sum() / prev_eq)
        size_frac = np.abs(values) / prev_eq
        rloss = max(0.0, 1.0 - prev_eq / eq_hist[max(0, len(eq_hist) - 21)])
        pos = np.sign(shares).astype(int)
        pnl = np.where(pos != 0, pos * (bt["close"][k] / np.where(entry_px > 0, entry_px, 1.0) - 1.0), 0.0)
        ctx = {"portfolio_drawdown": pdd, "gross_exposure": gross_now, "size_frac": size_frac, "recent_loss": rloss}
        probs = np.asarray(decide_fn(k, bt["rows"][k], pos, pnl, hold, ctx), dtype=float)       # PROPOSAL at the close of bar k
        valid = valid_action_mask(pos, allow_short)
        proposed = np.where(valid, probs, -1.0).argmax(axis=1)
        conf = probs[np.arange(S), proposed]
        reasons = np.zeros((S, len(REASON_NAMES)), dtype=bool)
        if gate is not None:                                                                    # the gate has final authority
            feats = {n: RAW[n][bt["rows"][k]] for n in REQUIRED_FEATURES}
            action, reasons = gate.review_batch(proposed, conf, pos, pnl, hold, feats,
                                                {"portfolio_drawdown": pdd, "gross_exposure": gross_now, "entry_fraction": entry_frac, "recent_loss": rloss})
            reason_counts += reasons.sum(axis=0)
            changed = np.where(action != proposed)[0]
            override_days.append(len(changed))
            for s in changed:
                events.append({"k": k, "symbol": bt["symbols"][s], "row": int(bt["rows"][k][s]), "proposed": int(proposed[s]), "final": int(action[s]),
                               "reasons": [REASON_NAMES[j] for j in np.where(reasons[s])[0]]})
        else:
            action = proposed
        o = bt["open"][k + 1]; traded = 0.0; entered = np.zeros(S, dtype=bool)
        for s in np.where(action == EXIT)[0]:                                                   # exits first: free capital
            sh = shares[s]
            px = o[s] * (1 - SLIP) if sh > 0 else o[s] * (1 + SLIP)
            cash += sh * px - FEE * abs(sh) * px
            traded += abs(sh) * px
            forced = [REASON_NAMES[j] for j in np.where(reasons[s])[0] if REASON_NAMES[j] in FORCED_EXIT_REASONS]
            trades.append({"symbol": bt["symbols"][s], "side": int(np.sign(sh)), "entry_date": bt["dates"][entry_k[s]], "exit_date": bt["dates"][k + 1],
                           "decision_date": bt["dates"][entry_k[s] - 1], "bars": int(hold[s]), "ret": float(np.sign(sh) * (px / entry_px[s] - 1.0)),
                           "exit_by": ("gate: " + forced[0]) if forced else "policy", "entry_conf": float(entry_conf[s])})
            shares[s] = 0.0; entry_px[s] = 0.0
        equity_open = cash + float((shares * o).sum())
        for s in [i for i in np.argsort(-conf) if action[i] in (BUY, SELL)]:                    # entries, most confident first
            scale = float(np.clip((conf[s] - 0.5) / 0.4, 0.5, 1.0)) if conf_sizing else 1.0
            target = entry_frac * equity_open * scale
            room = cap_gross * equity_open - float(np.abs(shares * o).sum())
            if action[s] == BUY:
                px = o[s] * (1 + SLIP); notional = min(target, room, cash / (1 + FEE))
            else:
                px = o[s] * (1 - SLIP); notional = min(target, room)
            if notional < CONFIG["min_trade_fraction"] * equity_open:
                n_skipped += 1; continue
            if action[s] == BUY:
                shares[s] = notional / px; cash -= notional * (1 + FEE)
            else:
                shares[s] = -notional / px; cash += notional * (1 - FEE)
            entry_px[s] = px; entry_k[s] = k + 1; entry_conf[s] = conf[s]; entered[s] = True; traded += notional; n_entries += 1
        c1 = bt["close"][k + 1]
        short_mask = shares < 0
        cash -= float(-(shares[short_mask] * c1[short_mask]).sum()) * BORROW_DAILY
        eq = max(cash + float((shares * c1).sum()), 1e-9)
        hold = np.where(entered, 1, np.where(shares != 0, hold + 1, 0))
        ret_hist.append(eq / prev_eq - 1.0); eq_hist.append(eq); eq_out.append(eq / CAP0)
        expo_hist.append(float(np.abs(shares * c1).sum()) / eq); turn_hist.append(traded / prev_eq)
        pdd_hist.append(max(0.0, 1.0 - eq / max(eq_hist[-(DD_LOOKBACK + 1):])))
    tr = pd.DataFrame(trades)
    return {"dates": bt["dates"][1:], "ret": np.array(ret_hist), "equity": np.array(eq_out), "exposure": np.array(expo_hist),
            "turnover": np.array(turn_hist), "portfolio_drawdown": np.array(pdd_hist), "n_entries": n_entries, "n_closed_trades": len(tr),
            "n_skipped_entries": n_skipped, "open_positions_at_end": int((shares != 0).sum()), "trades": tr,
            "win_rate": float((tr["ret"] > 0).mean()) if len(tr) else float("nan"),
            "avg_trade_return": float(tr["ret"].mean()) if len(tr) else float("nan"),
            "gate_reason_counts": dict(zip(REASON_NAMES, reason_counts.tolist())), "gate_events": events,
            "gate_override_days": np.array(override_days) if gate is not None else np.zeros(T - 1, dtype=int)}

def perf_metrics(r, turnover=None, exposure=None) -> dict:
    r = np.asarray(r, dtype=float); n = len(r); eq = np.cumprod(1.0 + r)
    years = n / 252.0; sd = r.std(ddof=1) if n > 1 else np.nan
    ex = r - CONFIG["risk_free_annual"] / 252.0
    dd_sd = np.sqrt((np.minimum(ex, 0.0) ** 2).mean()) if n else np.nan
    peak = np.maximum.accumulate(np.concatenate([[1.0], eq]))[1:]
    dd = eq / peak - 1.0; mdd = float(dd.min())
    cagr = float(eq[-1] ** (1 / years) - 1) if years > 0 and eq[-1] > 0 else float("nan")
    var5 = float(np.percentile(r, 5)) if n else float("nan")
    out = {"n_days": int(n), "total_return": float(eq[-1] - 1.0), "cagr": cagr,
           "ann_vol": float(sd * np.sqrt(252)) if sd == sd else float("nan"),
           "sharpe": float(ex.mean() / sd * np.sqrt(252)) if sd and sd > 0 else float("nan"),
           "sortino": float(ex.mean() / dd_sd * np.sqrt(252)) if dd_sd and dd_sd > 0 else float("nan"),
           "max_drawdown": mdd, "calmar": float(cagr / abs(mdd)) if mdd < 0 and cagr == cagr else float("nan"),
           "downside_deviation_ann": float(dd_sd * np.sqrt(252)) if dd_sd == dd_sd else float("nan"),
           "var_95_daily": var5, "cvar_95_daily": float(r[r <= var5].mean()) if n and (r <= var5).any() else float("nan"),
           "worst_day": float(r.min()) if n else float("nan"), "ulcer_index": float(np.sqrt(np.mean((100 * dd) ** 2))),
           "share_days_underwater": float((dd < 0).mean())}
    out["annualized_turnover"] = float(np.mean(turnover) * 252.0) if turnover is not None else 0.0
    out["avg_gross_exposure"] = float(np.mean(exposure)) if exposure is not None else 1.0
    out["max_gross_exposure"] = float(np.max(exposure)) if exposure is not None else 1.0
    return out

def _ctx_args(ctx, n):
    return {"portfolio_drawdown": ctx["portfolio_drawdown"], "gross_exposure": ctx["gross_exposure"],
            "position_size": ctx["size_frac"], "recent_loss": ctx["recent_loss"]}

def model_decide(profile):
    prof = np.asarray(profile, dtype=np.float32)
    def fn(k, rows, pos, pnl, hold, ctx):
        seq = RAW_MAT[rows[:, None] + OFFSETS_NP]                              # raw past-only windows (S, L, 43)
        return policy.predict({"sequence": seq, "position": pos, "profile": prof, "unrealized_pnl": pnl, "holding_bars": hold,
                               **_ctx_args(ctx, len(rows))})["action_probs"]
    return fn

def label_rule_decide(profile, allow_short):
    """The conservative objective applied directly (decision-time information only)."""
    prof = np.asarray(profile, dtype=np.float32)
    def fn(k, rows, pos, pnl, hold, ctx):
        port = {"pdd": ctx["portfolio_drawdown"], "gross": ctx["gross_exposure"], "size": ctx["size_frac"], "rloss": ctx["recent_loss"]}
        y = conservative_rule_labels({c: v[rows] for c, v in RAW.items()}, pos, np.tile(prof, (len(rows), 1)), hold, pnl, port, allow_short)
        return np.eye(N_ACTIONS)[y]
    return fn

def rule_risk_controller_decide(k, rows, pos, pnl, hold, ctx):
    """Simple rule-based risk controller (long-only): BUY when flat if SMA50 > SMA200, close > SMA200, volatility percentile < 0.8 and
    60-bar drawdown shallower than 8%; EXIT if SMA50 < SMA200, volatility percentile >= 0.9 or 60-bar drawdown deeper than 12%."""
    vp = 0.5 * (RAW["atr_pct_pctile_252"][rows] + RAW["rv20_pctile_252"][rows])
    y = np.zeros(len(rows), dtype=int)
    with np.errstate(invalid="ignore"):
        y[(pos == 0) & (RAW["sma50_sma200"][rows] > 0) & (RAW["close_sma200"][rows] > 0) & (vp < 0.8) & (RAW["dd_from_high_60"][rows] > -0.08)] = BUY
        y[(pos == 1) & ((RAW["sma50_sma200"][rows] < 0) | (vp >= 0.9) | (RAW["dd_from_high_60"][rows] < -0.12))] = EXIT
    return np.eye(N_ACTIONS)[y]

def always_long_decide(k, rows, pos, pnl, hold, ctx):
    y = np.zeros(len(rows), dtype=int); y[pos == 0] = BUY
    return np.eye(N_ACTIONS)[y]

LS = "long/short" if CONFIG["allow_short"] else "long-only"
MAIN = f"SP-05 policy + gate (balanced, {LS})"
LONG_ONLY = "SP-05 policy + gate (balanced, long-only)"
NOGATE = f"SP-05 policy ALONE, no gate (balanced, {LS})"
RULECTL = "Rule-based risk controller + gate (long-only)"
GATEONLY = "Always-long + gate (gate-only baseline)"
LABELRULE = f"Conservative label rule + gate (balanced, {LS})"
BT, BT_RESULTS, BT_TABLE = {}, {}, {}
for split in ("val", "test"):
    bt = prepare_backtest(split); BT[split] = bt
    runs = {}
    for pname, prof in PROFILES.items():
        runs[f"SP-05 policy + gate ({pname}, {LS})"] = run_portfolio_backtest(bt, model_decide(prof), CONFIG["allow_short"], GATE, CONFIG["confidence_sizing"])
    if CONFIG["allow_short"]:
        runs[LONG_ONLY] = run_portfolio_backtest(bt, model_decide(PROFILES["balanced"]), False, GATE, CONFIG["confidence_sizing"])
    runs[NOGATE] = run_portfolio_backtest(bt, model_decide(PROFILES["balanced"]), CONFIG["allow_short"], None, CONFIG["confidence_sizing"])
    runs[RULECTL] = run_portfolio_backtest(bt, rule_risk_controller_decide, False, GATE, False)
    runs[GATEONLY] = run_portfolio_backtest(bt, always_long_decide, False, GATE, False)
    runs[LABELRULE] = run_portfolio_backtest(bt, label_rule_decide(PROFILES["balanced"], CONFIG["allow_short"]), CONFIG["allow_short"], GATE, False)
    dates = runs[MAIN]["dates"]
    b_ret = bt["bench_close"][1:] / bt["bench_close"][:-1] - 1.0
    ew_ret = (bt["close"][1:] / bt["close"][:-1] - 1.0).mean(axis=1)
    metrics = {n: perf_metrics(r["ret"], r["turnover"], r["exposure"]) for n, r in runs.items()}
    metrics[f"{CONFIG['benchmark']} buy & hold"] = perf_metrics(b_ret)
    metrics["Equal-weight universe buy & hold"] = perf_metrics(ew_ret)
    EXPO_MATCHED = f"{CONFIG['benchmark']} scaled to the main run's average exposure (rest in cash)"
    _me = float(np.mean(runs[MAIN]["exposure"])); metrics[EXPO_MATCHED] = perf_metrics(b_ret * _me, None, np.full(len(b_ret), _me))      # shows how much of the 'safety' is just low exposure
    BT_RESULTS[split] = {"runs": runs, "bench_ret": b_ret, "ew_ret": ew_ret, "dates": dates}
    extra = pd.DataFrame({n: {"trade_entries": r["n_entries"], "closed_trades": r["n_closed_trades"], "skipped_entries": r["n_skipped_entries"],
                              "win_rate": r["win_rate"], "avg_trade_return": r["avg_trade_return"], "open_at_end": r["open_positions_at_end"],
                              "gate_overrides": int(sum(r["gate_override_days"]))} for n, r in runs.items()}).T
    BT_TABLE[split] = pd.DataFrame(metrics).T.join(extra)
    print(f"\n===== {split.upper()} backtest: {dates[0].date()} -> {dates[-1].date()} ({len(dates)} days, {len(bt['symbols'])} symbols, capital {CAP0:,.0f}) =====")
    print(f"fee {CONFIG['fee_bps']} bps + slippage {CONFIG['slippage_bps']} bps per side | entry size {CONFIG['position_fraction']:.0%} | gate max gross {RISK_LIMITS.max_gross_exposure:.0%} | shorts allowed: {CONFIG['allow_short']}")
    print(BT_TABLE[split][["total_return", "sharpe", "max_drawdown", "cvar_95_daily", "worst_day", "annualized_turnover", "avg_gross_exposure",
                           "trade_entries", "gate_overrides"]].round(3).to_string())


In [ ]:
# ---- What did the gate do? Overrides by reason, and (hindsight, evaluation only) what the blocked entries would have earned --------------
print("Gate overrides by reason (TEST), per run:")
gate_tab = pd.DataFrame({n: r["gate_reason_counts"] for n, r in BT_RESULTS["test"]["runs"].items() if n != NOGATE}).T
print(gate_tab.loc[:, (gate_tab != 0).any(axis=0)].to_string())

BLOCK_REASONS = {"risk_data_missing", "drawdown_halt", "recent_loss_halt", "volatility_prohibited", "abnormal_gap_or_move", "market_stress",
                 "low_confidence_entry", "gross_exposure_limit", "short_not_allowed"}
def blocked_entry_analysis(events):
    rows_ = []
    for e in events:
        if e["proposed"] in (BUY, SELL) and e["final"] == HOLD:
            f = FWD_RET[e["row"]]
            if np.isfinite(f):
                for rsn in (e["reasons"] or ["(other)"]):
                    if rsn in BLOCK_REASONS or rsn == "(other)":
                        rows_.append({"reason": rsn, "dir_adj_fwd_return": (1 if e["proposed"] == BUY else -1) * f})
    if not rows_:
        return pd.DataFrame()
    d = pd.DataFrame(rows_)
    return d.groupby("reason")["dir_adj_fwd_return"].agg(blocked_entries="size", mean_dir_adj_fwd_return="mean", share_that_would_have_lost=lambda s: (s < 0).mean())
BLOCKED = {s: blocked_entry_analysis(BT_RESULTS[s]["runs"][MAIN]["gate_events"]) for s in ("val", "test")}
print(f"\nBlocked entries of the main run and their {H}-bar forward return (hindsight, evaluation only; negative = the gate avoided a loss):")
for s in ("val", "test"):
    print(f"[{s}]"); print(BLOCKED[s].round(4).to_string() if len(BLOCKED[s]) else "no blocked entries")

# forced exits: the policy had not de-risked before a hard limit was hit
FORCED = {}
for s in ("val", "test"):
    tr = BT_RESULTS[s]["runs"][MAIN]["trades"]
    if len(tr):
        FORCED[s] = tr.groupby("exit_by").agg(trades=("ret", "size"), mean_return=("ret", "mean"), win_rate=("ret", lambda x: (x > 0).mean()), median_bars=("bars", "median"))
        print(f"\n[{s}] main-run closed trades by who exited them (gate-forced exits = cases the policy did not de-risk in time):")
        print(FORCED[s].round(4).to_string())

### 9c. Stress tests
**Past-only stress definitions on the benchmark** (known at the previous close, so no hindsight): *high volatility* = SPY 20-bar realised-volatility percentile ≥ 0.85 over its trailing 252 bars; *severe drawdown* = SPY more than 10% below its 252-bar high; *stress* = either. Returns are grouped by the flag known at the **previous** close. The tables compare the strategies in stress and normal periods on the validation and test splits (pooled and separately), plus days on which SPY fell sharply, and **arithmetic gap-loss budgets** (observed exposure × a hypothetical overnight gap: not a forecast, not a simulation of a crash).

In [ ]:
bench_feat = build_features(bench_df, bench_df)
BENCH_STRESS = pd.DataFrame({"high_vol": bench_feat["bench_rv20_pctile_252"] >= 0.85, "severe_dd": bench_feat["dd_from_high_252"] <= -0.10})
BENCH_STRESS["stress"] = BENCH_STRESS["high_vol"] | BENCH_STRESS["severe_dd"]
STRESS_STRATS = [MAIN, NOGATE, RULECTL, GATEONLY, f"{CONFIG['benchmark']} buy & hold", "Equal-weight universe buy & hold"]

def collect(split):
    R = BT_RESULTS[split]; flags = BENCH_STRESS.reindex(BT[split]["dates"][:-1]).fillna(False).astype(bool)
    series = {n: R["runs"][n]["ret"] for n in STRESS_STRATS if n in R["runs"]}
    series[f"{CONFIG['benchmark']} buy & hold"] = R["bench_ret"]; series["Equal-weight universe buy & hold"] = R["ew_ret"]
    expo = {n: R["runs"][n]["exposure"] for n in STRESS_STRATS if n in R["runs"]}
    return series, expo, flags

def group_stats(r, e, m):
    if m.sum() < 3:
        return None
    rr = r[m]; eq = np.cumprod(1 + rr); pk = np.maximum.accumulate(np.concatenate([[1.0], eq]))[1:]
    v5 = np.percentile(rr, 5)
    return {"days": int(m.sum()), "compounded_return": float(eq[-1] - 1), "mean_daily_%": float(100 * rr.mean()), "ann_vol": float(rr.std(ddof=1) * np.sqrt(252)),
            "worst_day": float(rr.min()), "cvar5_daily": float(rr[rr <= v5].mean()), "max_dd_within_group": float((eq / pk - 1).min()),
            "avg_gross_exposure": float(np.mean(e[m])) if e is not None else 1.0}

STRESS = {}
for scope in ("val", "test", "val+test"):
    parts = [collect(s) for s in (("val", "test") if scope == "val+test" else (scope,))]
    rows_ = []
    for name in STRESS_STRATS:
        r = np.concatenate([p[0][name] for p in parts if name in p[0]])
        e = np.concatenate([p[1][name] for p in parts]) if name in parts[0][1] else None
        fl = pd.concat([p[2] for p in parts])
        for gname, m in (("normal", ~fl["stress"].to_numpy()), ("high_vol", fl["high_vol"].to_numpy()), ("severe_drawdown", fl["severe_dd"].to_numpy()), ("any stress", fl["stress"].to_numpy())):
            st = group_stats(r, e, m)
            if st:
                rows_.append({"strategy": name, "period": gname, **st})
    STRESS[scope] = pd.DataFrame(rows_)
    print(f"\n===== STRESS ({scope}): returns by benchmark stress state known at the previous close =====")
    t = STRESS[scope]
    if len(t):
        print(t.pivot(index="strategy", columns="period", values="compounded_return").reindex(STRESS_STRATS).round(3).to_string())
        print("worst day by period:"); print(t.pivot(index="strategy", columns="period", values="worst_day").reindex(STRESS_STRATS).round(4).to_string())
        print("average gross exposure by period (does the system actually de-risk when stress is flagged?):")
        print(t.pivot(index="strategy", columns="period", values="avg_gross_exposure").reindex(STRESS_STRATS).round(3).to_string())
    else:
        print("no stress days in this window (normal for a calm sample): stress conclusions are not possible for it")

# ---- sharp benchmark down days (pooled val+test) -----------------------------------------------------------------
rets_all, spy_all = {n: [] for n in STRESS_STRATS if n in BT_RESULTS["test"]["runs"]}, []
for s in ("val", "test"):
    R = BT_RESULTS[s]; spy_all.append(R["bench_ret"])
    for n in rets_all: rets_all[n].append(R["runs"][n]["ret"])
spy_all = np.concatenate(spy_all); DOWN = {}
for thr in (-0.02, -0.03):
    m = spy_all <= thr
    if m.sum() >= 3:
        DOWN[f"SPY <= {thr:.0%}"] = {"days": int(m.sum()), "SPY mean return": float(spy_all[m].mean()),
                                    **{n: float(np.concatenate(v)[m].mean()) for n, v in rets_all.items()}}
DOWN_T = pd.DataFrame(DOWN).T
print("\nMean strategy return on days when SPY fell sharply (pooled val+test); capture = strategy mean / SPY mean (lower = better protection):")
if len(DOWN_T):
    cap = DOWN_T.drop(columns=["days", "SPY mean return"]).div(DOWN_T["SPY mean return"], axis=0)
    print(DOWN_T.round(4).to_string()); print("capture ratio:"); print(cap.round(2).to_string())
else:
    print("fewer than 3 such days: not enough information")

# ---- gap-loss budgets (arithmetic on observed exposure) ------------------------------------------------------------
GAP = {}
for n in (MAIN, NOGATE, RULECTL, GATEONLY):
    e = np.concatenate([BT_RESULTS[s]["runs"][n]["exposure"] for s in ("val", "test")])
    GAP[n] = {"mean_exposure": e.mean(), "p95_exposure": np.percentile(e, 95), "max_exposure": e.max(),
              **{f"loss_at_max_exposure_gap_{int(g*100)}%": -g * e.max() for g in (0.05, 0.10, 0.20)}}
GAP_T = pd.DataFrame(GAP).T
print("\nGap-loss budget: loss of equity if ALL gross exposure (assumed long) gapped down overnight. No controller can prevent this once exposure is open:")
print(GAP_T.round(3).to_string())

### 9d. Failure analysis: where did the system fail to avoid losses?

In [ ]:
def drawdown_episodes(eq, dates, top=3):
    eq = np.asarray(eq, dtype=float); peak = np.maximum.accumulate(np.concatenate([[1.0], eq]))[1:]
    dd = eq / peak - 1.0; eps, in_dd = [], False
    for t in range(len(eq)):
        if dd[t] < 0 and not in_dd:
            in_dd, start, trough = True, max(t - 1, 0), t
        if in_dd:
            if dd[t] < dd[trough]:
                trough = t
            if dd[t] >= 0:
                eps.append((start, trough, t)); in_dd = False
    if in_dd:
        eps.append((start, trough, None))
    rows_ = []
    for s, tr_, e in sorted(eps, key=lambda x: dd[x[1]])[:top]:
        rows_.append({"peak_date": dates[s].date(), "trough_date": dates[tr_].date(), "depth": float(dd[tr_]),
                      "days_peak_to_trough": int(tr_ - s), "recovered_on": dates[e].date() if e is not None else "NOT RECOVERED",
                      "days_underwater": int((e if e is not None else len(eq) - 1) - s)})
    return pd.DataFrame(rows_)

R = BT_RESULTS["test"]; dts = R["dates"]; main = R["runs"][MAIN]
named = {MAIN: main["ret"], NOGATE: R["runs"][NOGATE]["ret"], RULECTL: R["runs"][RULECTL]["ret"], f"{CONFIG['benchmark']} buy & hold": R["bench_ret"]}
FAIL = {"drawdown_episodes": {}, "calendar_years": None, "worst_days": None, "worst_21d": {}, "worst_trades": None, "forced_exits": FORCED.get("test")}
for name, r in named.items():
    FAIL["drawdown_episodes"][name] = drawdown_episodes(np.cumprod(1 + r), dts)
    print(f"\n--- worst drawdown episodes: {name} ---"); print(FAIL["drawdown_episodes"][name].to_string(index=False))
yearly = pd.DataFrame({n: pd.Series(r, index=dts).groupby(dts.year).apply(lambda x: np.prod(1 + x) - 1) for n, r in named.items()})
FAIL["calendar_years"] = yearly
print("\ncalendar-year compounded returns (partial years are partial):"); print(yearly.round(3).to_string())
for name, r in named.items():
    w = pd.Series(r).rolling(21).apply(lambda x: np.prod(1 + x) - 1, raw=True).dropna()
    FAIL["worst_21d"][name] = float(w.min()) if len(w) else float("nan")
print("worst 21-day compounded return:", {k: round(v, 4) for k, v in FAIL["worst_21d"].items()})

wd = pd.DataFrame({"date": dts, "main_return": main["ret"], "spy_return": R["bench_ret"], "gross_exposure": main["exposure"],
                   "portfolio_drawdown": main["portfolio_drawdown"], "gate_overrides_that_day": main["gate_override_days"]})
FAIL["worst_days"] = wd.nsmallest(5, "main_return")
print("\nworst 5 days of the main run (exposure and SPY move that day; losses with high exposure = failure to de-risk):")
print(FAIL["worst_days"].round(4).to_string(index=False))
tr = main["trades"]
if len(tr):
    FAIL["worst_trades"] = tr.nsmallest(5, "ret")[["symbol", "side", "decision_date", "exit_date", "bars", "ret", "exit_by"]]
    print("\nworst 5 closed trades of the main run:"); print(FAIL["worst_trades"].to_string(index=False))
    stop_hits = int(tr["exit_by"].str.contains("position_stop_loss").sum())
    print(f"\nhard stop-loss exits (gate had to force them): {stop_hits} of {len(tr)} closed trades; "
          f"max portfolio drawdown of the main run: {main['portfolio_drawdown'].max():.2%} "
          f"(halt at {RISK_LIMITS.halt_drawdown:.0%}, flatten at {RISK_LIMITS.flatten_drawdown:.0%})")
else:
    print("\nno closed trades in the test window for the main run")
print("\nCompare policy+gate with the 'ALONE' run: any loss the gate removed is the gate's merit, any loss that remains in 'policy + gate' is a failure of BOTH. "
      "Do not read a good calm-period result as protection; read the stress and failure tables.")


## 10. Plots

In [ ]:
FIG_DIR = ART_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history_df["epoch"], history_df["train_loss"], label="train loss")
ax[0].plot(history_df["epoch"], history_df["val_loss"], label="validation loss")
ax[0].axvline(BEST_EPOCH, color="grey", ls="--", lw=1, label=f"best epoch ({BEST_EPOCH})")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("weighted cross-entropy"); ax[0].set_title("Learning curves"); ax[0].legend()
ax[1].plot(history_df["epoch"], history_df["val_macro_f1"], color="tab:green")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("macro F1"); ax[1].set_title("Validation macro-F1")
plt.tight_layout(); plt.savefig(FIG_DIR / "learning_curves.png", dpi=130); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
cm = np.array(CLS["test"]["confusion_matrix"]); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
for a, mat, title, fmt in ((ax[0], cm, "Test confusion matrix (counts)", "d"), (ax[1], cmn, "Test confusion matrix (row-normalised)", ".2f")):
    a.imshow(mat, cmap="Blues"); a.set_title(title)
    a.set_xticks(range(N_ACTIONS)); a.set_yticks(range(N_ACTIONS)); a.set_xticklabels(ACTION_NAMES); a.set_yticklabels(ACTION_NAMES)
    a.set_xlabel("predicted"); a.set_ylabel("true (label)")
    for i in range(N_ACTIONS):
        for j in range(N_ACTIONS):
            a.text(j, i, format(mat[i, j], fmt), ha="center", va="center", color="white" if mat[i, j] > mat.max() / 2 else "black")
plt.tight_layout(); plt.savefig(FIG_DIR / "confusion_matrix_test.png", dpi=130); plt.show()

fig, ax = plt.subplots(figsize=(4.8, 4.5))
bins = pd.DataFrame(CLS["test"]["reliability_bins"])
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
ax.plot(bins["confidence"], bins["accuracy"], "o-", label=f"test (ECE {CLS['test']['ece']:.3f})")
ax.set_xlabel("mean confidence"); ax.set_ylabel("accuracy vs proxy labels"); ax.set_title("Reliability (vs labels, not vs loss avoidance)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "reliability_test.png", dpi=130); plt.show()

R = BT_RESULTS["test"]; main = R["runs"][MAIN]; d = R["dates"]
fl_plot = BENCH_STRESS.reindex(BT["test"]["dates"][:-1]).fillna(False).astype(bool)["stress"].to_numpy()
def shade_stress(a):
    start = None
    for i, v in enumerate(list(fl_plot) + [False]):
        if v and start is None: start = i
        if not v and start is not None:
            a.axvspan(d[start], d[min(i, len(d) - 1)], color="red", alpha=0.10, lw=0); start = None
fig, ax = plt.subplots(4, 1, figsize=(11, 13), sharex=True, gridspec_kw={"height_ratios": [3, 1.6, 1.4, 1.2]})
for a in ax: shade_stress(a)
ax[0].plot(d, main["equity"], label=MAIN, lw=2)
ax[0].plot(d, R["runs"][NOGATE]["equity"], label=NOGATE, lw=1.2)
ax[0].plot(d, R["runs"][RULECTL]["equity"], label=RULECTL, lw=1.2)
ax[0].plot(d, R["runs"][GATEONLY]["equity"], label=GATEONLY, lw=1.0)
ax[0].plot(d, np.cumprod(1 + R["bench_ret"]), label=f"{CONFIG['benchmark']} buy & hold", lw=1.2)
ax[0].plot(d, np.cumprod(1 + R["ew_ret"]), label="Equal-weight universe buy & hold", lw=1.2)
ax[0].set_ylabel("growth of 1.0"); ax[0].set_title("TEST split (out-of-sample): equity after costs. Red shading = benchmark stress (high vol or severe drawdown)"); ax[0].legend(fontsize=7)
for name, eq in ((MAIN, main["equity"]), (NOGATE, R["runs"][NOGATE]["equity"]), (CONFIG["benchmark"], np.cumprod(1 + R["bench_ret"]))):
    ax[1].plot(d, eq / np.maximum.accumulate(eq) - 1, label=name)
ax[1].axhline(-RISK_LIMITS.halt_drawdown, color="orange", ls="--", lw=0.8, label="gate: halt entries"); ax[1].axhline(-RISK_LIMITS.flatten_drawdown, color="red", ls="--", lw=0.8, label="gate: flatten")
ax[1].set_ylabel("drawdown"); ax[1].legend(fontsize=7)
ax[2].plot(d, main["exposure"], color="tab:purple", label="policy + gate"); ax[2].plot(d, R["runs"][NOGATE]["exposure"], color="tab:gray", lw=0.8, label="policy alone")
ax[2].set_ylabel("gross exposure"); ax[2].legend(fontsize=7)
ax[3].bar(d, main["gate_override_days"], color="tab:red", width=1.5); ax[3].set_ylabel("gate overrides/day")
plt.tight_layout(); plt.savefig(FIG_DIR / "backtest_test.png", dpi=130); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
gt = gate_tab.loc[[n for n in gate_tab.index if "policy + gate" in n or "Rule-based" in n or "Always" in n or "label rule" in n], :]
gt = gt.loc[:, (gt != 0).any(axis=0)]
if gt.shape[1]:
    gt.T.plot(kind="barh", ax=ax[0]); ax[0].set_title("TEST: gate overrides by reason"); ax[0].legend(fontsize=6); ax[0].set_xlabel("count")
st = STRESS["val+test"]
if len(st):
    pv = st.pivot(index="period", columns="strategy", values="compounded_return")[[c for c in STRESS_STRATS if c in st["strategy"].values]]
    pv.plot(kind="bar", ax=ax[1]); ax[1].axhline(0, color="k", lw=0.8); ax[1].set_title("val+test: compounded return by stress state"); ax[1].legend(fontsize=6); plt.setp(ax[1].get_xticklabels(), rotation=0)
plt.tight_layout(); plt.savefig(FIG_DIR / "gate_and_stress.png", dpi=130); plt.show()

fig, ax = plt.subplots(figsize=(7, 3.8))
(dist / dist.sum() * 100).plot(kind="bar", ax=ax); ax.set_ylabel("% of samples"); ax.set_title("Label distribution"); plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig(FIG_DIR / "label_distribution.png", dpi=130); plt.show()

## 11. Save standardized artifacts
| File | Content |
|---|---|
| `conservative_policy.pt` | weights + config (incl. the hard risk limits) + preprocessing + feature schema (self-contained, `weights_only=True`) |
| `conservative_config.json` | model / inference configuration, confidence limitations and risk limits (side-car copy) |
| `feature_schema.json` | exact input ordering and dimensions, adapter description, cross-policy compatibility block and the authority statement |
| `preprocessing.pkl` | train-fit median / mean / std / clip (plain dict, no custom classes) |
| `evaluation_metrics.json` | classification, risk-condition, gate, stress, failure and backtest metrics, with the proxy-label warning |
| `equity_curve.csv` | chronological equity curves (validation and test, each starting at 1.0) with exposure, drawdown and stress flags |
| `sp05_features.py`, `sp05_risk.py`, `sp05_model.py` | the exact code needed: features + adapter, deterministic gate, policy |

`conservative_policy.pt` is authoritative; the other copies are for inspection and tooling.

In [ ]:
def to_py(o):
    if isinstance(o, pd.DataFrame): return to_py(o.reset_index().to_dict(orient="records"))
    if isinstance(o, dict): return {str(k): to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_py(v) for v in o]
    if isinstance(o, np.ndarray): return to_py(o.tolist())
    if isinstance(o, (np.floating, float)): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, (pd.Timestamp,)) or hasattr(o, "isoformat"): return str(o)
    return o

pt_path = policy.save(ART_DIR)       # conservative_policy.pt, conservative_config.json, feature_schema.json, preprocessing.pkl

def strip_pred(c): return {k: v for k, v in c.items() if k != "pred"}
evaluation = {
    "model": "SP-05 Conservative Risk-Aware Policy", "label_source": LABEL_SOURCE, "label_warning": policy_config["label_note"],
    "scores_interpretation": ("Classification scores measure imitation of a rule-derived label set (strategy imitation), not human behaviour, not safety and not profitability."
                              if LABEL_SOURCE.startswith("strategy_derived_proxy") else "Classification scores measure behaviour cloning of the supplied trader actions."),
    "disclaimer": "Research and paper-trading system only. No capital protection is claimed; backtests are one historical path and are not evidence of future results.",
    "authority": policy_config["authority"], "confidence_limitations": policy_config["confidence_limitations"],
    "objective": CONFIG["objective"], "soft_limits": CONFIG["soft_limits"], "hard_risk_limits": RISK_LIMITS.to_dict(),
    "data": policy_config["data"], "n_samples": {k: int(len(v["y"])) for k, v in SAMPLES.items()},
    "label_distribution": dist.to_dict(), "class_weights": dict(zip(ACTION_NAMES, CLASS_WEIGHTS.tolist())),
    "classification": {"validation": strip_pred(CLS["val"]), "test": strip_pred(CLS["test"]), "test_uncalibrated": strip_pred(CLS_UNCAL_TEST)},
    "classification_by_risk_condition": COND_CLS, "entry_proposals_in_forbidden_states": VIOLATIONS,
    "backtest_assumptions": {"execution": "decision at close t, fill at open t+1", "fee_bps": CONFIG["fee_bps"], "slippage_bps": CONFIG["slippage_bps"],
                             "short_borrow_bps_annual": CONFIG["short_borrow_bps_annual"], "risk_free_annual": CONFIG["risk_free_annual"],
                             "allow_short": CONFIG["allow_short"], "initial_capital": CONFIG["initial_capital"],
                             "position_fraction": CONFIG["position_fraction"], "confidence_sizing": CONFIG["confidence_sizing"],
                             "max_position_size": CONFIG["max_position_size"], "max_gross_exposure_without_gate": CONFIG["max_gross_exposure"],
                             "baseline": "rule-based risk controller (long-only) run through the same gate; always-long + gate; SPY and equal-weight buy & hold"},
    "backtest": {s: BT_TABLE[s].to_dict(orient="index") for s in ("val", "test")},
    "gate_overrides_by_reason_test": gate_tab, "blocked_entries_hindsight": BLOCKED, "trades_by_exit_authority": FORCED,
    "stress_tests": {k: v for k, v in STRESS.items()}, "spy_down_days": DOWN_T, "gap_loss_budget": GAP_T,
    "failure_analysis_test": FAIL,
    "training_history": history_df.to_dict(orient="records"), "best_epoch": BEST_EPOCH,
}
(ART_DIR / "evaluation_metrics.json").write_text(json.dumps(to_py(evaluation), indent=2))

curves = []
for split in ("val", "test"):
    R_ = BT_RESULTS[split]; m_ = R_["runs"][MAIN]
    fl_ = BENCH_STRESS.reindex(BT[split]["dates"][:-1]).fillna(False).astype(bool)
    curves.append(pd.DataFrame({"date": R_["dates"], "split": split, "strategy_return": m_["ret"], "strategy_equity": m_["equity"],
                                "no_gate_equity": R_["runs"][NOGATE]["equity"], "rule_controller_equity": R_["runs"][RULECTL]["equity"],
                                "gate_only_equity": R_["runs"][GATEONLY]["equity"], "benchmark_return": R_["bench_ret"],
                                "benchmark_equity": np.cumprod(1 + R_["bench_ret"]), "equal_weight_bh_equity": np.cumprod(1 + R_["ew_ret"]),
                                "gross_exposure": m_["exposure"], "portfolio_drawdown": m_["portfolio_drawdown"], "turnover": m_["turnover"],
                                "gate_overrides": m_["gate_override_days"], "benchmark_stress_prev_close": fl_["stress"].to_numpy()}))
pd.concat(curves, ignore_index=True).to_csv(ART_DIR / "equity_curve.csv", index=False)

for fname in ("sp05_features.py", "sp05_risk.py", "sp05_model.py"):
    shutil.copy(fname, ART_DIR / fname)

# deterministic, download-free reload and gate reconstruction from the saved file
test_rows = eligible_rows("test")
reloaded = ConservativePolicy.load(ART_DIR, DEVICE)
_r = test_rows[:256]; _win = RAW_MAT[_r[:, None] + OFFSETS_NP]
_state = {"sequence": _win, "position": 0, "profile": PROFILES["balanced"]}
_a, _b = policy.predict(_state), reloaded.predict(_state)
assert np.allclose(_a["action_probs"], _b["action_probs"], atol=1e-6) and (_a["action"] == _b["action"]).all(), "reload mismatch!"
_split = {"sequence": _win[:, :, :N_BASE_FEATURES], "risk_features": _win[:, :, N_BASE_FEATURES:], "position": 0, "profile": PROFILES["balanced"]}
assert np.allclose(reloaded.predict(_split)["action_probs"], _b["action_probs"], atol=1e-6), "adapter form differs from extended form"
assert np.array_equal(reloaded.predict(_state)["action_probs"], reloaded.predict(_state)["action_probs"]), "inference is not deterministic"
_held = {**_state, "position": 1, "unrealized_pnl": -0.02, "holding_bars": 6, "portfolio_drawdown": 0.04, "gross_exposure": 0.3, "position_size": 0.1, "recent_loss": 0.01}
assert np.allclose(policy.predict(_held)["action_probs"], reloaded.predict(_held)["action_probs"], atol=1e-6)
assert RiskController.from_policy_config(reloaded.config).limits.to_dict() == RISK_LIMITS.to_dict(), "risk limits did not round-trip"
print("Round-trip check passed: reloaded policy reproduces predictions deterministically and the gate limits survive the round trip.")

ARTIFACTS = ["conservative_policy.pt", "conservative_config.json", "feature_schema.json", "preprocessing.pkl",
             "evaluation_metrics.json", "equity_curve.csv", "sp05_features.py", "sp05_risk.py", "sp05_model.py"]
for a in ARTIFACTS:
    assert (ART_DIR / a).exists(), f"missing artifact {a}"
    print(f"saved {(ART_DIR / a).resolve()}  ({(ART_DIR / a).stat().st_size / 1024:.1f} KiB)")

shutil.make_archive(str(ART_DIR), "zip", ART_DIR)
print(f"zip bundle: {Path(str(ART_DIR) + '.zip').resolve()}")
try:
    from google.colab import files  # noqa: F401  (only exists in Colab)
    files.download(str(ART_DIR) + ".zip")
except Exception:  # noqa: BLE001
    print("(not running in Colab or download blocked: copy the zip from the path above)")

## 12. Inference demonstration
Loads the saved policy **from disk**, builds a state with the documented adapter, gets a **proposal** from the network and then sends it through the **independent deterministic gate**. No data is downloaded and nothing is trained. The second and third examples show the gate overruling the network.

In [ ]:
policy_loaded = ConservativePolicy.load(ART_DIR, DEVICE)     # or ConservativePolicy.load("path/to/conservative_policy.pt", "cpu")
controller = RiskController.from_policy_config(policy_loaded.config)     # limits come from the saved config; no network involved
assert "yfinance" not in "".join(Path(f).read_text() for f in ("sp05_model.py", "sp05_features.py", "sp05_risk.py")), "inference modules must not depend on a data downloader"

sym0 = list(raw_data)[0]; ohlcv_hist = raw_data[sym0]
shared = {"sequence": build_features(ohlcv_hist, bench_df)[BASE_FEATURE_COLUMNS].iloc[-L:].to_numpy(dtype=np.float32),   # shared 30 columns
          "position": 0, "profile": {"risk_tolerance": 0.3, "patience": 0.7, "aggressiveness": 0.3}}
state = RiskFeatureAdapter().adapt_state(shared, ohlcv_hist, bench_df)                 # adds 'risk_features' (L, 13)
FN = list(FEATURE_COLUMNS)

def show(title, st):
    out = policy_loaded.predict(st)                                                      # PROPOSAL from the network
    prop = {"action": out["action"], "confidence": out["confidence"]}
    ful = {**{k: v for k, v in st.items() if k != "risk_features"}, "sequence": np.concatenate([np.asarray(st["sequence"]), np.asarray(st["risk_features"])], axis=1)}
    verdict = controller.review(prop, ful, FN)                                           # FINAL AUTHORITY: deterministic gate
    print(f"{title}\n   policy proposes {ACTION_NAMES[out['action']]} (confidence {out['confidence']:.3f}) -> gate decides {ACTION_NAMES[verdict['action']]}"
          f"{' (OVERRIDE: ' + ', '.join(verdict['reasons']) + ')' if verdict['overridden'] else ''}  | reasons logged: {verdict['reasons']}")
    return out, verdict

show(f"{sym0} @ {ohlcv_hist.index[-1].date()} | flat, normal portfolio", state)
show("same bar, portfolio is 9% below its peak (hard halt is 8%)", {**state, "portfolio_drawdown": 0.09, "gross_exposure": 0.2})
o_, v_ = show("long position, unrealized loss of 6.5% (hard stop is 5%)", {**state, "position": 1, "unrealized_pnl": -0.065, "holding_bars": 8, "position_size": 0.1})
assert v_["action"] == EXIT and "position_stop_loss" in v_["reasons"], "the hard stop must force EXIT whatever the network proposes"
print("   -> the stop-loss exit is enforced by plain code; the network's proposal was irrelevant to it")

# the gate is deterministic code: demonstrate with a deliberately adversarial proposal
adv = controller.review({"action": BUY, "confidence": 0.99}, {**{k: v for k, v in state.items() if k != "risk_features"},
                                                                 "sequence": np.concatenate([state["sequence"], state["risk_features"]], axis=1),
                                                                 "portfolio_drawdown": 0.20, "gross_exposure": 0.1}, FN)
print("adversarial BUY proposal at 20% portfolio drawdown ->", ACTION_NAMES[adv["action"]], adv["reasons"])
assert adv["action"] == HOLD

r = test_rows[:5]; w = RAW_MAT[r[:, None] + OFFSETS_NP]
batch = policy_loaded.predict({"sequence": w[:, :, :N_BASE_FEATURES], "risk_features": w[:, :, N_BASE_FEATURES:], "position": [0, 0, 1, 1, -1],
                               "profile": PROFILES["conservative"], "unrealized_pnl": [0, 0, 0.02, -0.03, 0.01], "holding_bars": [0, 0, 5, 9, 4],
                               "portfolio_drawdown": 0.02, "gross_exposure": 0.25, "position_size": [0, 0, 0.1, 0.1, 0.1], "recent_loss": 0.0})
print("\nbatch proposals:", batch["action_name"], "| confidences:", [round(float(c), 3) for c in batch["confidence"]])
try:
    policy_loaded.predict(shared)
except ValueError as exc:
    print("Expected guard when risk features are missing ->", str(exc)[:130], "...")

## 13. Assumptions, limitations and Meta-RL integration contract

In [ ]:
SP01_FEATURE_COLUMNS = ['ret_1', 'logret_1', 'ret_5', 'ret_10', 'ret_20', 'ret_60', 'rsi_14', 'macd_pct', 'macd_signal_pct', 'macd_hist_pct', 'close_ema20', 'close_ema50', 'ema20_ema50', 'close_sma200', 'sma50_sma200', 'adx_14', 'di_diff', 'atr_pct', 'rv_20', 'rv_60', 'vol_chg_1', 'vol_ratio_20', 'vol_z_60', 'hl_range', 'gap_open', 'close_loc', 'rel_strength_20', 'rel_strength_60', 'bench_ret_20', 'bench_rv_20']
SP04_STATIC = ['pos_flat', 'pos_long', 'pos_short', 'risk_tolerance', 'patience', 'aggressiveness', 'unrealized_pnl_scaled', 'holding_bars_scaled']
checks = {
    "action mapping is HOLD/BUY/SELL/EXIT = 0/1/2/3": ACTION_NAMES == ["HOLD", "BUY", "SELL", "EXIT"],
    "first 30 sequence columns identical to SP-01..SP-04 (order and names)": FEATURE_COLUMNS[:len(SP01_FEATURE_COLUMNS)] == SP01_FEATURE_COLUMNS,
    "first 8 static columns identical to SP-02..SP-04": STATIC_FEATURE_NAMES[:len(SP04_STATIC)] == SP04_STATIC,
    "adapter reproduces the training features exactly": True,   # asserted in section 4
    "loaded policy reproduces in-memory predictions deterministically": bool(np.allclose(_a["action_probs"], _b["action_probs"], atol=1e-6)),
    "hard limits round-trip through the saved config": True,   # asserted in section 11
    "gate forces EXIT at the hard stop and blocks entries during a portfolio-drawdown halt regardless of the proposal": True,   # asserted in section 12
    "inference modules contain no data downloader": "yfinance" not in "".join(Path(f).read_text() for f in ("sp05_model.py", "sp05_features.py", "sp05_risk.py")),
    "SELL is invalid while long/short; EXIT invalid while flat": bool(valid_action_mask([1], True)[0].tolist() == [True, False, False, True]
                                                                       and valid_action_mask([0], True)[0].tolist() == [True, True, True, False]),
}
assert all(checks.values()), f"compatibility check failed: {[k for k, v in checks.items() if not v]}"

print("=" * 100)
print("ARTIFACT PATHS")
for a in ARTIFACTS:
    print(f"  {(ART_DIR / a).resolve()}")
print(f"  {Path(str(ART_DIR) + '.zip').resolve()}   (zip of everything above)")
print("=" * 100)
print("ASSUMPTIONS")
print(f"  - data: yfinance daily adjusted OHLCV for {len(raw_data)} symbols + benchmark {CONFIG['benchmark']}; splits train<= {CONFIG['train_end']} < val <= {CONFIG['val_end']} < test")
print(f"  - labels: {LABEL_SOURCE}; decision-time information only, no future price in any label")
print(f"  - execution: decision at close t, fill at open t+1; fee {CONFIG['fee_bps']} bps + slippage {CONFIG['slippage_bps']} bps per side; short borrow {CONFIG['short_borrow_bps_annual']} bps/yr")
print(f"  - hard limits: {RISK_LIMITS.to_dict()}")
print(f"  - sizing: entry {CONFIG['position_fraction']:.0%} of equity x confidence factor ({CONFIG['confidence_sizing']}); shorts allowed: {CONFIG['allow_short']}")
print("KNOWN LIMITATIONS")
print("  - 'Conservative' is a design intent, not a guarantee: no capital protection is claimed. Gap losses on open exposure cannot be prevented by any controller (see the gap-loss budget).")
print("  - Daily bars: intraday moves, halts, liquidity and borrow availability are not modelled; stops execute at the next open, so realised losses can exceed the stop level.")
print("  - Confidence is calibrated against proxy labels on one validation period; it is not validated for stress states and is not comparable across policies.")
print("  - Proxy labels encode a human-written objective with explicit weights; the network imitates it and can only be as good as that objective. Weights were not optimised or tuned for returns.")
print("  - Portfolio inputs in the training data are sampled, not simulated, so the network has seen drawdown/exposure states only approximately.")
print("  - One symbol list of liquid survivors, one test regime, no walk-forward retraining; stress periods are few, so stress conclusions are weak.")
print("  - A conservative system also gives up returns: compare 'policy + gate' with buy-and-hold in strong up-markets as well as in stress.")
print("=" * 100)
print("META-RL INTEGRATION CONTRACT: SP-05 Conservative Risk-Aware Policy   ([x] = verified by an assertion in this notebook)")
for k in checks:
    print(f"  [x] {k}")
print( "  [ ] AUTHORITY: SP-05 only PROPOSES. Route every action (from ANY policy, including the selector's own choice) through RiskController.review_batch / review;")
print( "      the controller is deterministic code with no learned parameters and may override HOLD/BUY/SELL, but never blocks an EXIT proposal.")
print( "  [ ] Build limits from the saved config: controller = RiskController.from_policy_config(policy.config); keep the same limits in paper trading and in backtests.")
print( "  [ ] Send the SHARED state to every policy; for SP-05 add the adapter output and portfolio numbers:")
print( "        state = RiskFeatureAdapter().adapt_state(shared_state, ohlcv_history, benchmark_history)   # >= seq_len + 290 bars, no download")
print( "        state.update(portfolio_drawdown=..., gross_exposure=..., position_size=..., recent_loss=...)   # fractions of equity, all optional (default 0)")
print( "  [ ] Track in the environment: side, entry fill price (-> unrealized_pnl fraction), bars held (entry bar = 1), portfolio equity peak, gross exposure, ~20-bar equity change.")
print( "  [ ] policy.predict(state) -> action_probs(4), action, action_name, confidence, valid_actions, entropy   (a PROPOSAL)")
print( "  [ ] Execution timing: decide at the close of bar t, fill at the open of bar t+1; EXIT is the only way to close; SELL only opens a short (if allow_short).")
print(f"  [ ] allow_short = {CONFIG['allow_short']} in this artifact; the gate's allow_short must match the execution layer.")
print( "  [ ] Log every gate override with its reason (the backtest does) and review them: a rising override rate means the policy is drifting from the limits.")
print( "  [ ] Do NOT use SP-05's confidence to relax any limit, and do not compare it with other policies' confidences.")
print( "  [ ] Re-run with walk-forward retraining before trusting any single split; keep this system to research and paper trading.")
print("-" * 100)
print("OUT-OF-SAMPLE (TEST) SNAPSHOT: one regime, one symbol list, costs included; NOT proof of profitability or of capital protection")
test_m = BT_TABLE["test"]
snap_rows = [MAIN] + ([LONG_ONLY] if LONG_ONLY in test_m.index else []) + [NOGATE, RULECTL, GATEONLY, f"{CONFIG['benchmark']} buy & hold", f"{CONFIG['benchmark']} scaled to the main run's average exposure (rest in cash)", "Equal-weight universe buy & hold"]
print(test_m.loc[snap_rows, ["total_return", "sharpe", "max_drawdown", "cvar_95_daily", "avg_gross_exposure", "annualized_turnover", "trade_entries"]].round(3).to_string())
print("-" * 100)
print("Research and paper-trading system only. Not a recommendation to trade real money. No profitability or capital protection is guaranteed or implied.")
print("=" * 100)